# Notebook 06 -- score_one_model

**Score the whole corpus with ONE language model.** Run this notebook once per
model on the roster: set `MODEL_KEY`, run it start to finish, come back, change
that one line, run it again.

This is the notebook that spends money, and it is built around two facts. The
calls are paid, and a Colab runtime is dropped without warning. So every batch of
answers is written to disk the moment it comes back, and a re-run picks up where
the last one stopped instead of buying the same scores a second time.

**Reads** `openends_long.csv` (notebook 00) and `rubric.json` (notebook 05).
**Writes** `llm_scores_<MODEL_KEY>.jsonl`, one line per call carrying the model's
own reply text beside the record the parser made of it -- that pair is the raw
output prereg 6.7 asks to be kept "so the scoring can be repeated" -- and
`llm_scores_<MODEL_KEY>.csv`, one row per response x item with one column per
facet.

## Config

In [1]:
# === CONFIG: the only cell you edit ===
# Every switch in the pipeline lives here. Nothing below this cell needs changing.

RUN_MODE    = "full"        # "smoke" = first 20 rows, costs pennies, proves the
                            #           whole chain works end to end.
                            # "full"  = the real corpus.
HUMAN_GOLD  = False         # False = the pre-registered pipeline. Language models
                            #         do the coding, exactly as prereg 6.7 says.
                            #         This is the default and it runs start to
                            #         finish with no human step.
                            # True  = ALSO run the human-coder arm. That arm is a
                            #         DEVIATION from the registration and needs the
                            #         OSF addendum posted first.
WORKDIR     = "/content/retribution_nlp"   # where every file in the chain lives
RANDOM_SEED = 20260629      # frozen. Changing it re-draws every random split.

# --- PER-NOTEBOOK: the three lines below differ in each notebook ---
NB_ID       = "06"                     # this notebook's number
NB_NAME     = "score_one_model"        # this notebook's job, in two words
NB_ASKS     = "Score the whole corpus with one language model."

## Setup

In [2]:
# === SETUP: folders, the manifest, and the two functions every notebook uses ===
# WHAT THIS CELL DOES: makes the working folder, loads the manifest, and defines
# need() and done(). You never edit this cell.
#
# THE MANIFEST is a small file called _manifest.json that lives beside the data.
# Every notebook records what it produced in there. The next notebook reads it to
# check its inputs exist. That is what lets a notebook tell you "run 03 first"
# instead of dying on a missing file three cells later.

import os, json, hashlib, datetime, sys

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
MANIFEST = os.path.join(WORKDIR, "_manifest.json")

def _load_manifest():
    """Read the manifest, or start an empty one on the first notebook."""
    if os.path.exists(MANIFEST):
        with open(MANIFEST) as f:
            return json.load(f)
    return {"files": {}, "runs": []}

def _sha256(path, cap=8_000_000):
    """First 8 MB is plenty to catch a file that changed underneath us."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(cap))
    return h.hexdigest()[:16]

class MissingInput(Exception):
    """Raised when a required upstream file is not there yet."""
    pass

def need(filename, produced_by=None, optional=False):
    """Return the path to an input file, or explain which notebook makes it.

    This is the function that makes the chain navigable. A missing input stops
    the notebook HERE, with a sentence telling you what to run, rather than
    surfacing as a confusing error further down.
    """
    path = os.path.join(WORKDIR, filename)
    if os.path.exists(path):
        rec = _load_manifest()["files"].get(filename, {})
        stamp = rec.get("written_at", "unknown time")
        print(f"  [ok]   {filename}  (written {stamp})")
        return path
    if optional:
        print(f"  [skip] {filename} is not here. This notebook will carry on "
              f"without it and say so in its output.")
        return None
    where = f" Run notebook {produced_by} first." if produced_by else ""
    raise MissingInput(
        f"\n\n  MISSING INPUT: {filename}\n"
        f"  This notebook cannot run without it.{where}\n"
        f"  Files present in {WORKDIR}:\n    "
        + "\n    ".join(sorted(os.listdir(WORKDIR)) or ["(none)"]) + "\n")

def done(filename, rows=None, note=""):
    """Record a file this notebook produced, so later notebooks can find it."""
    path = os.path.join(WORKDIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"done() called for {filename} but it is not there")
    m = _load_manifest()
    m["files"][filename] = {
        "written_by": f"{NB_ID}_{NB_NAME}",
        "written_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "rows": rows,
        "sha16": _sha256(path),
        "run_mode": RUN_MODE,
        "note": note,
    }
    m["runs"] = [r for r in m["runs"] if r.get("nb") != NB_ID]
    m["runs"].append({"nb": NB_ID, "name": NB_NAME, "run_mode": RUN_MODE,
                      "human_gold": HUMAN_GOLD,
                      "at": datetime.datetime.now().isoformat(timespec="seconds")})
    with open(MANIFEST, "w") as f:
        json.dump(m, f, indent=1)
    size = os.path.getsize(path)
    print(f"  [wrote] {filename}  ({rows if rows is not None else '?'} rows, "
          f"{size:,} bytes)")

print(f"NOTEBOOK {NB_ID} -- {NB_NAME}")
print(f"  {NB_ASKS}")
print(f"  mode: RUN_MODE={RUN_MODE}, HUMAN_GOLD={HUMAN_GOLD}")
print(f"  folder: {WORKDIR}")
_m = _load_manifest()
if _m["runs"]:
    print("  already run in this folder: "
          + ", ".join(sorted(r["nb"] for r in _m["runs"])))
else:
    print("  this folder is empty: notebook 00 is the place to start.")

NOTEBOOK 06 -- score_one_model
  Score the whole corpus with one language model.
  mode: RUN_MODE=full, HUMAN_GOLD=False
  folder: /content/retribution_nlp
  this folder is empty: notebook 00 is the place to start.


## Load

One line to set, then two inputs, both fetched with `need()`:

- `MODEL_KEY` (the cell below) -- which one of the three raters this run is.
- `rubric.json` (notebook 05) -- the written codebook the models are given. Every
  word of every prompt sent below comes out of that file, so the raters cannot
  drift apart from each other: one codebook, three scorers. A leg given nudged
  wording would make inter-model disagreement partly prompt variance, and the
  agreement statistic would stop meaning inter-rater reliability.
- `openends_long.csv` (notebook 00) -- one row per participant per open-ended
  question, vignette columns already coalesced, 948 participants x 3 questions
  less the blanks. This notebook checks for itself that the file really does hold
  one row per response x item, because that pair is what everything downstream
  joins on.

Nothing in this notebook reads a human code, and `HUMAN_GOLD` never appears below
the setup cell. Prereg 6.7 has the models coding in place of human raters.

In [ ]:
# === PICK THE RATER: the one line in this notebook you edit ===================
# MODEL_KEY names the SINGLE rater this run scores the corpus with. Set it, run
# the notebook, then come back, change this one line, and run it again for the
# next model. Three runs of this notebook = the three-rater panel prereg 6.7
# requires.
#
# WHY ONE MODEL PER RUN. Three near-identical notebooks drift: a fix applied to
# the Claude copy and forgotten in the open-weight copy puts a difference in
# PROMPT or PARSING between two raters, and the inter-model agreement then partly
# measures our own editing rather than the raters. One notebook run three times
# cannot drift. It also means a provider outage costs one leg, not the batch.
#
# WHY IT SITS HERE AND NOT IN THE CONFIG CELL. That cell is byte-identical in all
# thirteen notebooks, which is what lets a reader who has understood one navigate
# the rest. Notebook 08 -- the other notebook that runs once per model -- puts its
# ENCODER_KEY in exactly this position, for exactly this reason.
MODEL_KEY = "claude"      # "claude" | "gpt" | "open" -- the roster is below

# === THE SECOND THING THIS NOTEBOOK NEEDS FROM YOU: WHERE THE CHECKPOINT LIVES
# WORKDIR is /content, which Colab wipes when the runtime is dropped, and what it
# wipes here is roughly an hour of PAID calls. The cell further down copies the
# archive to Drive as it goes and copies it BACK on startup, which is what makes
# "a re-run picks up where the last one stopped" true across a disconnect rather
# than only within one session.
#
# This is a deliberate exception to the Drive-free rule the rest of the chain
# follows. Everywhere else a lost session costs a free re-run; here it costs the
# leg. Set BACKUP_TO_DRIVE = False to keep the original behaviour, in which case
# the only protection is the browser download at the very end and an interrupted
# run loses everything since it started.
BACKUP_TO_DRIVE       = True
DRIVE_BACKUP_DIR      = "/content/drive/MyDrive/retribution_nlp_archive"
BACKUP_EVERY_BATCHES  = 5      # ~300 calls between copies, matching PRINT_EVERY


In [ ]:
# === INSTALL: the two client libraries Colab does not ship ===================
# WHY IT RUNS EVERY TIME: a Colab VM is discarded when the runtime disconnects,
# so the install happens at the top of every session. It computes nothing and
# writes no analysis file.
#
#   anthropic   the Claude leg.
#   openai      the second proprietary leg AND, through its OpenAI-compatible
#               endpoint, the hosted open-weight leg. One library, two raters.
# pandas, numpy and tqdm already ship in the Colab image.
#
# WHY THE COMPILED CORE IS NEVER PINNED HERE. numpy, pandas and scipy ship in the
# Colab image compiled against one another. A pip install that resolves a
# different numpy leaves the already-built wheels pointing at the wrong C struct
# layout, and the next import dies with
#     ValueError: numpy.dtype size changed, may indicate binary incompatibility
# Nothing here needs a different numpy, so nothing asks for one. Reproducibility
# is served by RECORDING what the session resolved (printed below, and archived
# with the outputs) rather than by forcing a downgrade that stops the notebook
# running at all.
#
# IF YOU EVER SEE THAT ERROR: pip cannot swap a compiled module the live kernel
# has already imported. Runtime -> Restart session, then run from the top.
# WHY THESE TWO *ARE* PINNED, WHEN THE COMPILED CORE ABOVE IS NOT.
# The rule is not "never pin", it is "pin what pinning cannot break". numpy and
# pandas ship compiled against the Colab image, so forcing a version there
# breaks the import. anthropic and openai are pure Python with no such coupling,
# and they are the two libraries whose behaviour IS the measurement: they carry
# the sampling settings prereg 6.7 fixes.
#
# THIS IS NOT HYPOTHETICAL. anthropic 1.0.0 reached PyPI on 20 August 2026 and
# REMOVED temperature, top_p and top_k from the Messages methods outright. The
# smoke test run on 24 August installed it unpinned, got 1.0.0, and every Claude
# call died before it left the machine with
#     TypeError: Messages.create() got an unexpected keyword argument 'temperature'
# reported as claude 0/20 while gpt and open both scored 20/20. Four days between
# a library release and a leg of this study silently ceasing to run.
#
# BOTH ARE PINNED TO THE EXACT VERSIONS NOTEBOOK 05 PROVED THE PANEL ON.
# 05's smoke test resolved anthropic 0.125.0 and openai 2.54.0 and got 20/20 on
# every leg; rubric.json records that pair under smoke_test.client_versions.
# These are the same two libraries, so this run scores the corpus with the exact
# stack the prompts were proven against.
#
# THE RANGE PIN IS NOT ENOUGH HERE, WHICH IS WHY THESE ARE EXACT. "anthropic<1.0"
# keeps the temperature parameter alive but still lets pip pick a different 0.x
# between one leg and the next, and this notebook is run three times over three
# sessions. Two legs on two client versions is a difference between raters that
# nobody chose and nothing downstream would surface.
!pip install -q "anthropic==0.125.0" "openai==2.54.0"

# Reproducibility record: the resolved versions belong in the OSF bundle next to
# the scores, so a later run can be compared against this one. The model version
# string matters more (it is pinned in the roster below), but a client library
# that changed its default behaviour is worth being able to see.
import importlib, sys
from importlib import metadata as _md

def _ver(mod_name, dist_name=None):
    # Not every package exposes __version__, so fall back to the installed
    # distribution metadata before giving up.
    try:
        return importlib.import_module(mod_name).__version__
    except Exception:
        pass
    try:
        return _md.version(dist_name or mod_name)
    except Exception as e:
        return f"NOT AVAILABLE -- {type(e).__name__}"

print("\nResolved versions (archive these with the outputs):")
print(f"  python     {sys.version.split()[0]}")
for _mod in ["numpy", "pandas", "anthropic", "openai"]:
    print(f"  {_mod:10s} " + str(_ver(_mod)))


Resolved versions (archive these with the outputs):
  python     3.13.15
  numpy      2.1.3
  pandas     2.2.3
  anthropic  0.125.0
  openai     2.54.0


In [ ]:
# === IMPORTS =================================================================
# WHAT THIS CELL DOES: brings in the names used below. Nothing on disk changes.
import pandas as pd
import numpy as np
# json      reading rubric.json, parsing the model replies, writing the archive
# re        regular expressions (patterns that match text): stripping markdown
#           fences off a reply, and repairing a truncated one, in parse_json_safe
# time      sleep between retries: a fixed pause on a parse failure, exponential
#           backoff on a rate limit
# threading a lock around the token counter, which eight worker threads all add to
import re, time, threading, warnings

# WHY THIS IS NARROW AND NOT A BARE "ignore". Silencing every category would
# silence the warnings a paid run most needs to see: a client library announcing
# that an option this notebook passes has been removed, or a provider warning
# that a reply came back cut short. RuntimeWarning -- numeric edge cases such as
# a division by zero in the progress arithmetic before the first batch has
# finished -- is the only category muted, and it can never carry a fact about
# the data.
warnings.filterwarnings("ignore", category=RuntimeWarning)

# The scoring loop is I/O-bound -- one HTTPS round trip per call -- so threads,
# not processes, are the right tool: eight in flight keeps the provider busy
# while the interpreter sits idle waiting on sockets.
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# Colab Secrets, so an API key is never typed into a cell and never lands in the
# .ipynb that goes to OSF.
from google.colab import userdata

print("Imports complete.")

Imports complete.


In [ ]:
# === THE RATER PANEL, AND THE PINNED VERSION OF EACH MODEL ====================
# WHAT THIS CELL DOES: names every model in the study with its exact dated version
# string. MODEL_KEY (config cell) picks the one THIS run uses; the others are
# listed so the roster is written down in one place and so the closing summary can
# say which raters are still outstanding.
#
# WHICH DESIGN DECISION IT IMPLEMENTS
#   Prereg 6.7: "Coding is done by an ensemble of language models in place of
#   human raters, at least three of them from different developers and including
#   one open-weight model." That sentence is a measurement requirement, not a
#   procurement note, and this dict is where it is either honoured or broken.
#
# THE THREE CONSTRAINTS, AND WHAT EACH ONE IS FOR (prereg 6.7; design section 4)
#   >=3 MODELS.     Two raters give an ICC -- an intraclass correlation, the
#                   statistic for how closely independent raters' numbers track
#                   one another -- but no majority: the holistic call has no
#                   tie-break, and one outlier moves the mean by half its
#                   distance. Three is the smallest panel where a single odd
#                   reading is visible AS an odd reading.
#   >=3 DEVELOPERS. The independence that matters is at the lab level, not the
#                   checkpoint level. Two snapshots from one lab share pretraining
#                   data, tokenizer, tuning policy and refusal behaviour, so they
#                   share blind spots; averaging them is closer to sampling one
#                   rater twice than to adding a rater. Three snapshots from one
#                   developer would post a high ICC and certify nothing.
#   >=1 OPEN-WEIGHT. Reproducibility (anyone holding the OSF bundle can re-run
#                   that leg from published weights) and diversity (a different
#                   pretraining mix and post-training regime, so its errors have
#                   the best chance of being uncorrelated with the proprietary
#                   pair). It is the leg the shared-error diagnostic leans on
#                   hardest, which is why every model's scores stay individually
#                   addressable in their own file all the way to the export.
#
# WHY A STRONGER ANTHROPIC TIER THAN THE DARK SIDE PIPELINE USED. Dark Side NB02
# scored a prosocial/dark binary with Haiku. Here the scorer has to hold desert
# apart from proportionality apart from censure apart from moral balance -- four
# boundaries a coarse valence read collapses (design section 1). It is a one-time
# batch of about 2,800 rows (948 participants x 3 questions, less the blanks), so
# the stronger tier costs little and buys the distinction the study rests on.
#
# PIN EXACT DATED STRINGS BEFORE THE PRODUCTION RUN. An alias such as
# 'claude-3-5-sonnet-latest' silently re-points at a new checkpoint, which makes
# the run unrepeatable and quietly invalidates the pinned-version clause of
# prereg 6.7. Confirm the IDs at run time: the Dark Side strings are ~1 yr old.
# See MEASUREMENT_DESIGN.md section 11 decision 1 for the open roster choices.
# NOT ONE OF THE THREE LEGS CARRIES A DATE SUFFIX, AND THAT IS NOT AN OVERSIGHT.
# Prereg 6.7 asks for a "pinned dated version string", which in the Dark Side era
# meant a form like 'claude-3-5-sonnet-20241022'. All three providers have since
# moved their current lines off dated snapshots:
#   claude-opus-4-6                          IS the complete Anthropic id.
#                                            Appending a date produces a 404.
#   gpt-5.6-terra                            is the explicit tier id. The BARE
#                                            alias 'gpt-5.6' routes to sol and is
#                                            a moving target -- never use it here.
#   meta-llama/Llama-3.3-70B-Instruct-Turbo  is the whole Together string.
# What each of these IS is explicit and non-routing, which is the property the
# clause was protecting: none is a '-latest' tag that silently re-points. The
# rest of the guarantee comes from the archive -- notebook 06 writes every model
# reply verbatim to llm_scores_<MODEL>.jsonl -- so a run stays auditable from its
# own output even if a provider later moves an id. Record this in the OSF
# deviation register: it is a change in what the industry offers, not a
# loosening on our side.
#
# WHY THIS PANEL, AND WHY EVERY LEG RUNS GREEDY.
# Prereg 6.7 fixes the sampling setting, and this study operationalised that as
# temperature 0. Holding that across all three legs is what drove each choice:
#   claude-opus-4-6  the strongest Anthropic tier that still ACCEPTS temperature.
#                    It was removed from Opus 4.7 onward, so claude-opus-5 would
#                    run leg 1 at the model default while legs 2 and 3 stayed
#                    greedy. The extra variance on one leg alone would surface as
#                    inter-model disagreement and depress the ICC -- reading as
#                    rater disagreement when it is really sampling noise.
#   gpt-5.6-terra    with reasoning_effort='none'. GPT-5.x accepts temperature
#                    ONLY at that setting; at any real reasoning effort it
#                    rejects it outright. The classic GPT-4.x models that took
#                    temperature unconditionally are being retired (gpt-4o
#                    2024-05-13 is already gone), so this is how this leg stays
#                    greedy, not a preference.
#   Llama-3.3-70B    takes temperature 0 unconditionally. Meta-developed, so it
#                    supplies the third developer AND the open-weight leg in one.
# The cost of this panel is that both proprietary legs run deliberately de-tuned:
# an Anthropic tier one generation back, and an OpenAI model with its reasoning
# switched off. That is the price of the registered fixed-setting clause, and it
# was paid deliberately.
MODELS = {
    'claude': 'claude-opus-4-6',                          # Anthropic. Takes temperature.
    'gpt':    'gpt-5.6-terra',                            # OpenAI. Needs effort='none'.
    'open':   'meta-llama/Llama-3.3-70B-Instruct-Turbo',  # Meta, open weights, Together.
}
# The open-weight leg is reached through an OpenAI-compatible hosted endpoint, so
# the same adapter serves it and the GPT leg: only the base URL and the model
# string move. Together is shown; swap the URL for Fireworks, vLLM or another
# host. Running the weights locally through transformers instead is the one
# configuration that needs a GPU runtime.
OPEN_BASE_URL = 'https://api.together.xyz/v1'
print(f"Roster declared: {len(MODELS)} raters, {', '.join(MODELS)}.")

Roster declared: 3 raters, claude, gpt, open.


In [ ]:
# === WHICH RATER THIS RUN IS, AND WHETHER IT IS PINNED =======================
# One lookup, and a sentence if it fails. A MODEL_KEY that is not on the roster
# would otherwise surface as a KeyError inside a worker thread, several cells and
# one API bill later.
if MODEL_KEY not in MODELS:
    raise KeyError(
        f"MODEL_KEY is '{MODEL_KEY}', which is not a rater on the roster. "
        f"Set it in the config cell to one of: {', '.join(MODELS)}.")
MODEL_ID = MODELS[MODEL_KEY]
print(f"This run scores with: {MODEL_KEY}  ->  {MODEL_ID}")
print("Roster (each needs its own run of this notebook):")
for _k, _v in MODELS.items():
    print(f"  {_k:8s} -> {_v}" + ("   <- this run" if _k == MODEL_KEY else ""))
# THIS NOTEBOOK REFUSES TO START ON AN UNPINNED MODEL. It used to print a NOTE
# and carry on. That is the wrong shape here for two reasons. This is the paid
# pass: an unpinned string means roughly 2,800 calls that each fail with
# model-not-found, retried five times apiece, and the failure is invisible until
# the run ends because the retry loop records an unparsed row rather than an
# error. And a run that DID somehow complete against a moving alias could not be
# reproduced, which is the clause prereg 6.7 is protecting. Refusing costs a
# re-run of one cell; not refusing costs the batch.
if "<" in MODEL_ID:
    raise ValueError(
        f"MODEL_ID for the '{MODEL_KEY}' leg is still the placeholder "
        f"'{MODEL_ID}'. Pin the exact snapshot in the roster cell above, then "
        "re-run. Nothing has been spent. (Prereg 6.7 requires a pinned version "
        "so the scoring can be repeated; see that cell for why the Anthropic "
        "leg carries no date suffix.)")

This run scores with: claude  ->  claude-opus-4-6
Roster (each needs its own run of this notebook):
  claude   -> claude-opus-4-6   <- this run
  gpt      -> gpt-5.6-terra
  open     -> meta-llama/Llama-3.3-70B-Instruct-Turbo


In [ ]:
# === HOW EVERY CALL IS MADE (identical for all three raters) ==================
# These settings are shared across the legs so that the only thing differing
# between raters is the rater.
#
# temperature 0.0  Scoring is measurement, not generation. At temperature above 0
#                  the same passage returns a different number on a re-run, which
#                  turns sampling noise into apparent inter-model disagreement and
#                  makes the ICC unrepeatable. Prereg 6.7's "fixed setting" is
#                  this. Greedy decoding is still not bit-identical across a
#                  provider's own infrastructure, which is exactly why all raw
#                  output is saved rather than assumed regenerable.
# max_tokens 900   Headroom for the full nine-facet object plus the holistic call
#                  plus optional evidence spans. Set it too low and long replies
#                  truncate mid-object; the parser repairs some of that, but a
#                  repaired object can be missing trailing facets, which land as
#                  NaN and read downstream as "facet absent" rather than
#                  "unmeasured".
GEN_KW = {'temperature': 0.0, 'max_tokens': 900}

# HAZARD -- two unrelated things in this pipeline are called a seed.
#   DECODE_SEED is the DECODING seed handed to OpenAI-compatible endpoints
#   (best-effort determinism on top of temperature 0; the Anthropic API ignores
#   it). RANDOM_SEED in the config cell is the frozen number that draws the
#   SEED/TEST halves in notebook 03. Changing DECODE_SEED re-rolls decoding ties
#   and costs nothing; the frozen split must never be re-rolled at all.
DECODE_SEED = 12345

# WHY 8 WORKERS. Enough parallelism to keep a provider busy, low enough to stay
# under a per-key rate limit. Raising it does not go faster once the provider
# starts returning "429 too many requests" -- it just pushes every worker into
# exponential backoff at the same moment. Lower it if this leg's throttling is
# aggressive.
MAX_WORKERS = 8

# The >=3-word floor, applied when the corpus is loaded and re-checked inside
# every scorer. Below three words there is no reasoning to score: "yes", "idk",
# "he did it" carry no facet structure, and a model asked to score them anyway
# returns confident noise that lands in the agreement statistics AS agreement.
MIN_WORDS = 3

# The passage window. The longest item (ranking) has a median of 32 words, so
# 1500 characters clears essentially every response in the corpus; the cap exists
# to stop one pathological entry from blowing the context or the bill. It is
# applied identically in the joint and the single-facet prompts, so both read the
# same passage -- otherwise the halo diagnostic in the E4 analysis would be
# comparing scores taken from two different texts.
TEXT_WINDOW = 1500
print(f"Call settings: temperature {GEN_KW['temperature']}, max_tokens "
      f"{GEN_KW['max_tokens']}, {MAX_WORKERS} calls in flight, "
      f"passage window {TEXT_WINDOW} characters.")

Call settings: temperature 0.0, max_tokens 900, 8 calls in flight, passage window 1500 characters.


In [ ]:
# === BATCHING AND RESUMING ===================================================
# BATCH_SIZE is how many calls are made before the results are written to disk.
# It is the single most important number in this notebook after MODEL_KEY.
#
# WHY CHECKPOINT AT ALL. The run is thousands of paid calls over hours, and a
# Colab VM is discarded on disconnect. Anything held only in memory is one
# dropped connection away from being bought twice. Writing after every batch
# means a run that dies at 80% keeps 80% of what it paid for, and the next run
# resumes from there.
# WHY 60 AND NOT 1. Writing after every single call would be safer by a few
# seconds of work and would spend the whole run opening and closing a file. Sixty
# calls is a couple of minutes of exposure at the worst possible moment.
BATCH_SIZE = 60
PRINT_EVERY = 5          # print a progress and cost line every N batches

# RETRY_FAILED decides what a resumed run does about calls that came back
# unparseable last time. False by default: a reply that failed five attempts at
# temperature 0 failed structurally, not transiently, and paying for it again on
# every resume buys the same failure. Set it True only after fixing the cause the
# failure log points at (a wrong model string, an unsupported option, a refusal).
# Note what True does to the archive: the re-run appends a SECOND line for that
# call, and the fold at the end of this notebook takes the later one. That is the
# behaviour to want, and it is counted and reported there rather than assumed.
RETRY_FAILED = False
print(f"Checkpoint every {BATCH_SIZE} calls. Failed calls are "
      + ("re-run" if RETRY_FAILED else "left as they are")
      + f" on a resume (RETRY_FAILED={RETRY_FAILED}).")

Checkpoint every 60 calls. Failed calls are left as they are on a resume (RETRY_FAILED=False).


In [ ]:
# === THE PRICE TABLE AND THE TOKEN COUNTER ===================================
# US dollars per MILLION tokens.
# ***TEAM: CHECK THESE AGAINST THE PROVIDER'S PRICE PAGE BEFORE THE RUN.***
# These numbers are used for one thing only -- the running cost line printed
# below -- so a stale figure misinforms the budget and can never touch a score.
# Input tokens are the prompt (the codebook plus the passage, which is most of
# the bill here); output tokens are the small JSON object that comes back.
# Checked against the provider price pages on 2026-08-24, for the exact roster
# pinned above. The previous figures were carried over from an earlier, cheaper
# roster and understated a full three-leg run by about $23: the Claude row was
# Sonnet-tier pricing while the pinned model is Opus 4.6, and the GPT row had
# the old output rate. A cost line that reads low is worse than none, because it
# is the number the budget decision gets made on.
PRICES = {
    'claude': {'in': 5.00,  'out': 25.00},   # claude-opus-4-6
    'gpt':    {'in': 2.50,  'out': 15.00},   # gpt-5.6-terra
    'open':   {'in': 1.04,  'out': 1.04},    # Llama-3.3-70B-Instruct-Turbo, Together
}

# The token counter every adapter adds to. It is touched by eight threads at
# once, so the additions happen under a lock: two threads incrementing the same
# integer without one can lose an update, and the cost estimate would drift low.
USAGE = {'in': 0, 'out': 0, 'calls': 0, 'estimated': 0}
_usage_lock = threading.Lock()

def _add_usage(n_in, n_out, estimated=False):
    """Record one call's token use. `estimated` counts the calls whose provider
    reported no token figures, so the printed cost can say how much of it is a
    guess instead of presenting a guess as a measurement."""
    with _usage_lock:
        USAGE['in'] += int(n_in or 0)
        USAGE['out'] += int(n_out or 0)
        USAGE['calls'] += 1
        if estimated:
            USAGE['estimated'] += 1

def record_usage(usage_obj, prompt, reply, in_name, out_name):
    """One call's tokens: the provider's own figures where it reports them, an
    estimate from length where it does not.

    WHY THE FALLBACK EXISTS. Providers differ in what they put in the usage
    envelope, and the hosted open-weight leg -- the one whose SDK shape varies
    most -- is the likeliest to send none at all. Counting those calls as zero
    tokens would print "$0.00 spent" across a run that was really buying
    thousands of calls, which is the one number in this notebook a reader would
    act on. Four characters per token is the usual English rule of thumb, and
    every estimated call is counted so the printed line can say how many."""
    n_in = getattr(usage_obj, in_name, None)
    n_out = getattr(usage_obj, out_name, None)
    if not n_in and not n_out:     # no usage object, or an empty envelope
        _add_usage(len(prompt or '') / 4, len(reply or '') / 4, estimated=True)
    else:
        _add_usage(n_in or 0, n_out or 0)

def cost_so_far():
    """Dollars spent on this leg so far, from the token counts above."""
    p = PRICES.get(MODEL_KEY, {'in': 0.0, 'out': 0.0})
    return USAGE['in'] / 1e6 * p['in'] + USAGE['out'] / 1e6 * p['out']

def usage_line():
    """The cost so far, and how much of it is measured rather than guessed."""
    if not USAGE['estimated']:
        return f"${cost_so_far():,.2f}"
    return (f"${cost_so_far():,.2f} [{USAGE['estimated']:,} of {USAGE['calls']:,} "
            f"calls reported no token count and were estimated from length]")

print(f"Prices used for the estimate: {PRICES.get(MODEL_KEY)} US dollars per "
      f"million tokens.")

Prices used for the estimate: {'in': 5.0, 'out': 25.0} US dollars per million tokens.


In [ ]:
# === THE API KEY FOR THIS ONE LEG, AND THE CLIENT IT BUILDS ===================
# WHY ONLY ONE KEY. This run scores with one model, so it loads one key. Loading
# all three would stop a colleague who has only the Claude key in their Colab
# Secrets, for a reason that has nothing to do with the leg they are running.
#
# KEYS COME FROM userdata.get, NEVER FROM A CELL. A key pasted into a cell is
# saved inside the .ipynb, and this .ipynb goes to OSF with the prompts and the
# raw output (prereg 6.7). Secrets keep the key out of the file. _load_secret
# raises rather than returning None, so a missing key fails here, in the first
# ten seconds, instead of surfacing as a wall of authentication failures that the
# retry loop swallows into unscored rows halfway through a paid corpus pass.
SECRET_NAMES = {
    'claude': 'Punishment_ClaudeAPI',
    'gpt':    'Punishment_OpenAIAPI',        # or Punishment_GeminiAPI if the
                                             # second-developer leg is Gemini
    'open':   'Punishment_OpenWeightAPI',
}

def _load_secret(name):
    # userdata.get returns an empty string (not an exception) when the secret
    # exists but is blank, and a blank key produces authentication failures that
    # look exactly like rate limits. Treat empty as fatal.
    try:
        val = userdata.get(name)
        if not val:
            raise ValueError("the secret is there but empty")
        # Last four characters only -- enough to tell two keys apart in the run
        # log, not enough to be a key.
        print(f"  {name} loaded (ends in ...{val[-4:]})")
        return val
    except Exception as e:
        print(f"  Could not load {name} from Colab Secrets: {e}")
        print(f"    Add it via the key icon in the left sidebar, then re-run.")
        raise

print("Loading the API key for this leg...")
_key = _load_secret(SECRET_NAMES[MODEL_KEY])

# One client object, constructed once and shared across the eight worker threads.
# All of these SDK clients are safe to call concurrently.
claude_client = gpt_client = open_client = None
if MODEL_KEY == 'claude':
    import anthropic
    # The pin is only real if the kernel picked it up. pip can install 0.x
    # while the live kernel still holds the 1.x it imported earlier in the
    # session; the install succeeds, the import succeeds, and then every call
    # dies on the temperature argument partway through a PAID pass.
    _am = str(getattr(anthropic, '__version__', '0')).split('.')[0]
    if _am != '0':
        raise RuntimeError(
            f'The kernel is holding anthropic {anthropic.__version__}, but this '
            'notebook needs 0.x.\n  1.0.0 removed the temperature parameter and '
            'prereg 6.7 fixes temperature at 0.\n  Fix: Runtime -> Restart '
            'session, then Run all. Nothing has been spent.')
    claude_client = anthropic.Anthropic(api_key=_key)
elif MODEL_KEY == 'gpt':
    from openai import OpenAI
    gpt_client = OpenAI(api_key=_key)
elif MODEL_KEY == 'open':
    from openai import OpenAI
    open_client = OpenAI(api_key=_key, base_url=OPEN_BASE_URL)
del _key          # the value stays inside the client object, not in a variable
                  # that a later print or traceback could expose
print(f"Client ready for the {MODEL_KEY} leg.")

Loading the API key for this leg...
  Punishment_ClaudeAPI loaded (ends in ...sAAA)
Client ready for the claude leg.


In [ ]:
# === LOAD THE CODEBOOK THE MODELS ARE GIVEN ==================================
# rubric.json holds the written instruction, the strict JSON schema, the three
# item prompts and the canonical facet vocabulary. It is read, never rewritten
# here: one codebook, three scorers, and the only way to change what a rater is
# asked is to change notebook 05 and re-run every leg.
rubric_path = need("rubric.json", produced_by="05")
with open(rubric_path) as fh:
    RUBRIC = json.load(fh)

def _rub(*names, required=True, default=None, what="", instead=""):
    """Pull one piece out of rubric.json, accepting any of several key spellings.

    A required piece that is missing stops the notebook here with a sentence,
    rather than sending thousands of paid calls carrying an empty instruction."""
    for n in names:
        if RUBRIC.get(n):
            return RUBRIC[n]
    if required:
        raise MissingInput(
            f"\n\n  rubric.json carries no {what}.\n"
            f"  Looked for the keys: {', '.join(names)}\n"
            f"  Keys it does have: {', '.join(sorted(RUBRIC))}\n"
            f"  Re-run notebook 05, which writes this file.\n")
    print(f"  [note] rubric.json carries no {what}. {instead}")
    return default

JSON_SYSTEM_PROMPT = _rub('system_prompt', 'json_system_prompt',
                          what="system prompt")
FACET_RUBRIC = _rub('facet_rubric', 'rubric_text', 'annotator_instruction',
                    what="nine-facet annotator instruction")
ITEM_PROMPTS = _rub('item_prompts', 'prompts',
                    what="three item prompts (backward / forward / ranking)")
# The canonical facet vocabulary. Order is fixed and this list is canonical:
# the prompts, the per-model columns and every downstream table iterate it, so
# renaming an entry in notebook 05 renames columns in every file at once. There
# is deliberately no fallback -- a rubric with no facet list is a broken rubric,
# and guessing the nine names here is how two notebooks quietly disagree about
# what "moral_balance" is called.
FACETS = _rub('facets', 'FACETS', 'facet_names', what="list of the nine facets")
# The single holistic call, kept OUT of the nine-facet vector: it comes from the
# same pass and the same rater as the facets, so it is descriptive coverage, never
# corroboration for facet-level co-occurrence (design section 9 rule 6).
HOLISTIC_LEVELS = _rub('holistic_levels', required=False,
                       default=['principled', 'vengeful', 'both', 'neither'],
                       what="list of holistic levels",
                       instead="Using the four standard ones.")

print(f"\nCodebook loaded: {len(FACETS)} facets, {len(ITEM_PROMPTS)} item prompts, "
      f"holistic levels {HOLISTIC_LEVELS}.")
print("  facets: " + ", ".join(FACETS))

  [ok]   rubric.json  (written unknown time)

Codebook loaded: 9 facets, 3 item prompts, holistic levels ['principled', 'vengeful', 'both', 'neither'].
  facets: desert, proportionality, censure, moral_balance, revenge, suffering, deterrence, incapacitation, rehabilitation


In [ ]:
# === IS THIS THE FACET LIST THE REST OF THE CHAIN EXPECTS? ===================
# CROSS-CHECK, and deliberately not a fallback. Notebook 07 carries its own
# hardcoded copy of the nine names (its cell "SETUP (cont.): imports, display
# width, and the facet vocabulary") and finds this notebook's columns by them.
# Nothing else in the chain compares the two lists. A facet renamed in notebook
# 05 would rename the columns written here and leave 07 looking for a name that
# no longer exists -- which 07 reports as a facet every rater failed to score,
# not as a name that moved. Said here, before any money is spent. It does not
# stop the run: rubric.json is the authority on what a facet is called.
FACETS_NB07 = ['desert', 'proportionality', 'censure', 'moral_balance', 'revenge',
               'suffering', 'deterrence', 'incapacitation', 'rehabilitation']
_drift = sorted(set(FACETS) ^ set(FACETS_NB07))
if _drift:
    print("\n  WARNING: rubric.json's facet names are not the nine notebook 07 "
          f"looks for.\n  Differing names: {', '.join(_drift)}."
          "\n  Update the list in notebook 07 to match before running it, or those "
          "facets\n  will read there as facets no rater scored.")

# === SETTINGS THE CODEBOOK CARRIES WITH IT ===================================
# rubric.json records the passage window, the word floor and the sampling
# settings, because each of them changes what a model sees or how it answers. If
# they are in the file, the file wins: the paid run must not quietly use a
# different window from the one the codebook was written and smoke-tested for.
# Where the file says nothing, the values set further up stand.
TEXT_WINDOW = int(RUBRIC.get('text_truncation_chars', TEXT_WINDOW))
MIN_WORDS   = int(RUBRIC.get('min_words', MIN_WORDS))
GEN_KW['max_tokens'] = int(RUBRIC.get('max_tokens', GEN_KW['max_tokens']))
GEN_KW['temperature'] = float(RUBRIC.get('temperature', GEN_KW['temperature']))
# The OpenAI leg's third fixed setting, read from the same record as the other
# two. Defaulting to 'none' rather than to the provider's default is deliberate:
# GPT-5.x accepts temperature ONLY at 'none', so a rubric.json written before
# this setting existed still yields a greedy leg 2 rather than a silently
# non-greedy one.
GPT_REASONING_EFFORT = str(RUBRIC.get('gpt_reasoning_effort', 'none'))

# The codebook's fingerprint: a short hash over exactly the text a model is shown
# -- system prompt, annotator instruction, schema, item prompts. It travels with
# every score this notebook writes, so that a later notebook comparing two legs
# can see at a glance whether they were scored under the same instruction. Two
# legs scored under different codebooks are not two raters of one thing, and
# their agreement would not mean what an agreement statistic is taken to mean.
RUBRIC_FINGERPRINT = RUBRIC.get('rubric_fingerprint', 'none recorded')

print(f"Codebook fingerprint: {RUBRIC_FINGERPRINT}")
print(f"  passage window {TEXT_WINDOW} characters, word floor {MIN_WORDS}, "
      f"temperature {GEN_KW['temperature']}, max_tokens {GEN_KW['max_tokens']}")
if GEN_KW['temperature'] != 0.0:
    print("  WARNING: temperature is not 0. Prereg 6.7 fixes the setting, and at "
          "any higher temperature the same passage returns a different number on "
          "a re-run, which turns sampling noise into apparent disagreement "
          "between the raters.")

Codebook fingerprint: fef0381e46892391
  passage window 1500 characters, word floor 3, temperature 0.0, max_tokens 900


In [ ]:
# === THE STRICT JSON SCHEMA SENT WITH EVERY JOINT CALL =======================
# Exactly nine facet keys, one holistic key, one evidence object. Sending the
# literal target shape, rather than describing it in prose, is what makes the
# reply parseable often enough that unscored rows stay rare.
#
# WHY `evidence` IS OPTIONAL, AND WHY THAT IS A MEASUREMENT DECISION. A mandatory
# verbatim span turns the scorer into a lexical matcher: it must find a quotable
# string before it can score a facet above zero. That suppresses exactly the
# non-literal vengeful and censure sentiment the ensemble exists to catch where
# the word list already fails -- "he should never see daylight again" is
# suffering with no suffering word in it. Making the field optional, and saying
# so in the user message, means the span requirement never quietly gates
# revenge/suffering recall (design section 4).
SCHEMA_HINT = _rub('schema_hint', 'schema', required=False, default=None,
                   what="strict JSON schema line",
                   instead="Building one from the facet list instead.")
if SCHEMA_HINT is None:
    # Built from FACETS so the fallback can never list a facet the codebook does
    # not have, or miss one it does.
    SCHEMA_HINT = ('{' + ','.join(f'"{f}":0.0' for f in FACETS)
                   + ',"holistic":"' + '|'.join(HOLISTIC_LEVELS) + '"'
                   + ',"evidence":{}}')
print("Schema sent with every joint call:")
print("  " + SCHEMA_HINT)

Schema sent with every joint call:
  {"desert":0.0,"proportionality":0.0,"censure":0.0,"moral_balance":0.0,"revenge":0.0,"suffering":0.0,"deterrence":0.0,"incapacitation":0.0,"rehabilitation":0.0,"holistic":"principled|vengeful|both|neither","evidence":{}}


In [ ]:
# === LOAD THE CORPUS, AND CHECK ITS ONE-ROW-PER-ANSWER CONTRACT ==============
corpus_path = need("openends_long.csv", produced_by="00")
df = pd.read_csv(corpus_path)

# Assert the three item types survived the export. The ranking item is the
# fragile one: the preregistration calls it Q333 while the Qualtrics export
# column is Priorit_Explain, and it is the only one of the three that is not
# split across vignettes. It is also the item E3 leans on hardest (design
# section 3), so a silent drop would remove the most direct elicitation of
# retributive reasoning and leave the analysis looking merely underpowered.
# Prereg 5.15 requires the three items be scored SEPARATELY and never
# concatenated; item_type is what carries that separation through every method.
assert set(df['item_type'].unique()) == {'ranking', 'backward', 'forward'}, \
    (f"Unexpected item_type set: {set(df['item_type'].unique())}. "
     f"Re-run notebook 00, which writes openends_long.csv.")

# The four columns every cell below reaches for. Checked here, by name, so a
# renamed column upstream says so in one line instead of surfacing as an
# AttributeError inside a worker thread halfway through a paid run.
_missing = [c for c in ('response_id', 'item_type', 'text', 'n_words')
            if c not in df.columns]
assert not _missing, (f"openends_long.csv is missing {_missing}. "
                      f"It has: {list(df.columns)}. "
                      f"Re-run notebook 00, which writes this file.")

# --- ONE ROW PER response_id x item_type, ENFORCED HERE ----------------------
# 948 participants x 3 open-ended questions, and the PAIR (response_id,
# item_type) is the key of the whole chain: every merge from here to the export
# joins on it. Notebook 00 checks this when it writes the file. It is checked
# AGAIN here, on the file as it actually arrives, because a corpus can be
# re-exported, hand-edited or concatenated with itself in between, and nothing
# would notice: a repeated pair multiplies rows through every later join, so the
# sample size of every downstream test grows while the printed participant count
# still reads 948, and every p-value in the exploratory family shrinks with it.
# WHAT MAKES IT FIRE: any repeated (response_id, item_type) pair in the file.
_dups = df.duplicated(['response_id', 'item_type'], keep=False)
assert not _dups.any(), (
    f"openends_long.csv holds {int(_dups.sum())} rows sharing a "
    f"(response_id, item_type) pair -- for example "
    f"{df.loc[_dups, ['response_id', 'item_type']].head(3).to_dict('records')}. "
    f"Everything below joins on that pair and must not fan out. Re-run notebook "
    f"00, which writes one row per participant per question.")
print(f"Corpus contract holds: {len(df):,} rows, one per response x item.")

  [ok]   openends_long.csv  (written unknown time)
Corpus contract holds: 2,844 rows, one per response x item.


In [ ]:
# === WHICH ROWS ARE SENT TO THE MODEL ========================================
# --- The >=3-word validity filter (carried over from Dark Side NB02) ---------
# Note what the filter does and does not do: it decides which rows are SENT to
# the model. It never deletes a row from the record -- every excluded row is
# still in openends_long.csv, and every parse failure among the rows that WERE
# sent is kept as an explicit unscored row (see below).
df_valid = df[df['n_words'] >= MIN_WORDS].reset_index(drop=True)

# --- Smoke mode: the first 20 PARTICIPANTS -----------------------------------
# A smoke run proves the whole chain -- key, prompt, parse, checkpoint, resume,
# CSV -- for the price of a couple of cups of coffee. The rows it scores are real
# scores at temperature 0, written to the same archive, so a later full run
# resumes over them and pays only for the remainder.
if RUN_MODE == "smoke":
    # Trim by PARTICIPANT, never by row. Every person contributes up to three
    # rows, one per open-ended question, and notebooks 00 and 03 both cut on
    # participants. A head(20) here would keep 20 ROWS, which is six and two
    # thirds of a person, and the three-rows-per-person check downstream would
    # fail on the participant left half in and half out.
    _smoke_ids = pd.unique(df_valid["response_id"])[:20]
    df_valid = df_valid[df_valid["response_id"].isin(_smoke_ids)].reset_index(drop=True)
    # The config cell calls smoke mode "first 20 rows" because that cell is
    # identical in all thirteen notebooks. Here it is 20 PEOPLE, and the true
    # size of the rehearsal is printed rather than described.
    print(f"\nSMOKE MODE: the first {len(_smoke_ids)} participants, which is "
          f"{len(df_valid)} rows, not 20 rows.")

print(f"\nCorpus: {len(df)} rows, {df['response_id'].nunique()} participants.")
print(f"Sent to the model: {len(df_valid)} rows (n_words >= {MIN_WORDS}), "
      f"{df_valid['response_id'].nunique()} participants.")
print(df_valid['item_type'].value_counts().to_string())
# WHAT THE PRINTOUT MEANS. The per-item counts should be roughly balanced across
# ranking, backward and forward. A count far below the other two means an export
# column did not coalesce upstream and that item is being lost -- not that
# participants skipped it.


SMOKE MODE: the first 20 participants, which is 59 rows, not 20 rows.

Corpus: 2844 rows, 948 participants.
Sent to the model: 59 rows (n_words >= 3), 20 participants.
item_type
ranking     20
backward    20
forward     19


## Do

First the four pieces that turn a passage into a score are built -- the prompt,
the parser, the adapter that talks to this run's provider, and the two scorers.
None of them sends anything: the cell that spends money is the batched run
further down, and it is the only one that does.

In [ ]:
# === THE SINGLE PROMPT-ASSEMBLY PATH (joint nine-facet call) =================
# Every joint-call score comes through this one function, on every leg, so there
# is exactly one place the instruction can change and no way for two raters to be
# asked different questions.
#
# The item prompt is passed as CONTEXT and immediately labelled context-only.
# Prereg 5.15 scores the three open-ends separately because the backward pair
# (why this sentence, why this ranking) and the forward item (what should it
# accomplish) elicit different registers, and that difference is itself a measure
# (E6). Telling the model which question the passage answers helps it read the
# passage; letting that fact push a label would score the question instead of the
# writer.
def build_user_prompt(text, item_type):
    # .get with an empty-string default rather than [] indexing: an unexpected
    # item_type degrades to "no prompt context" instead of raising inside a
    # worker thread, where the exception would become an unscored row.
    prompt_text = ITEM_PROMPTS.get(item_type, '')
    return (
        FACET_RUBRIC
        + f'\n\nThe writer was answering this prompt (context only; do NOT let it bias the labels): '
          f'"{prompt_text}"'
        # Triple quotes fence the passage so the model can tell instruction from
        # data. chr(34) * 3 builds them without nesting quote characters inside
        # an already quote-heavy f-string.
        + '\n\nPassage to score:\n' + (chr(34) * 3) + '\n' + text[:TEXT_WINDOW]
        + '\n' + (chr(34) * 3) + '\n\n'
        + 'Return ONLY a JSON object with EXACTLY these keys and 0.0-1.0 facet values '
          '(evidence is OPTIONAL; use <=12-word verbatim spans, and it is fine to omit a '
          'facet from evidence when it is present but not attributable to a single span):\n'
        + SCHEMA_HINT
    )

print("Joint-call prompt assembly ready. Example, first 300 characters:\n")
print(build_user_prompt("He deserves every year of it.", "backward")[:300] + " ...")

Joint-call prompt assembly ready. Example, first 300 characters:

You are one of several independent expert annotators coding how ordinary people justify a criminal sentence. Score each of NINE reasoning facets from 0.0 (absent) to 1.0 (clearly, centrally present) for how strongly the passage expresses it.

Score each facet INDEPENDENTLY. Several facets can be hig ...


### The E4 pair is elicited a second time, in separate calls

E4's hypothesis **is** the desert x revenge association: does principled desert
language appear alongside vengeful language in the same person's writing? In the
joint call both numbers come out of one read of one passage, sharing the rater's
holistic impression, the same 1,500-character window, and a rubric that
explicitly says "real explanations mix motives". A within-rater halo under a
co-occurrence instruction would produce a positive desert x revenge correlation
whether or not the writing contains one -- and that is double-dipping, since a
single joint read would then be producing both variables whose association is the
hypothesis.

So each of the two facets is scored again in its **own** call, with no
other-facet context, and then a third time with the co-occurrence sentence
removed. The joint scores are kept as well, and the difference between the
correlations is the halo diagnostic: a correlation that collapses under split
scoring was a rating artifact rather than a property of the text.

This is a registered deviation. Prereg 6.7 says "each model scores every response
on the facets" -- one pass, all facets -- so the split-call pass is an ADDITION
carried on the OSF addendum and listed in the deviation table in
MEASUREMENT_DESIGN.md section 0. The preregistered joint-call estimate is
reported in full alongside it; the split-call estimate carries the primary
weight for E4.

Cost: four extra calls per response on top of the one joint call. That is the
price of not letting a single read produce both sides of E4's hypothesis.

In [ ]:
# === THE SINGLE-FACET PROMPT, FOR desert AND revenge ONLY ====================
# The two definitions are worded to match the joint codebook exactly, so the only
# difference between a joint and a split score is the ISOLATION, not the
# construct. Note what is absent: neither definition mentions the other facet, so
# nothing in a desert call can prime a revenge reading or the reverse.
SPLIT_FACETS = ('desert', 'revenge')
SOLO_DEFN = _rub('solo_definitions', 'solo_defn', required=False, default=None,
                 what="single-facet definitions for the E4 pair",
                 instead="See what that means for the separate-call passes below.")

# The negation and stance teaching case is kept in BOTH variants. It is about
# reading the writer's stance correctly, not about facets co-occurring, so
# dropping it would degrade accuracy rather than test anchor sensitivity.
SOLO_TEACH = _rub('solo_teaching_case', 'solo_teach', 'solo_teaching', required=False,
    what="negation and stance sentence for the single-facet prompt",
    instead="Using the wording carried in this cell, so the prompt stays complete.",
    default=('Handle negation and stance: a passage that REJECTS or critiques this idea scores '
             'LOW; "he deserves NO mercy" is MAX-desert, not a negation. Do not infer the facet '
             'from the crime; score only what the TEXT expresses.'))
# This sentence -- and only this sentence -- is what the mixed_anchor switch
# toggles. It is the co-occurrence instruction: keep it and the split call still
# tells the model that motives mix; remove it and the model is given no reason at
# all to expect the other facet. Scoring both ways quantifies how much of any
# desert x revenge co-occurrence rides on the instruction rather than on the
# writing. A co-occurrence that depends on the rubric's own co-occurrence
# instruction is circular.
SOLO_COOCC = _rub('solo_cooccurrence_sentence', 'solo_cooccurrence',
    'mixed_anchor_sentence', required=False,
    what="co-occurrence sentence the mixed-anchor switch removes",
    instead="Using the wording carried in this cell, so both variants stay defined.",
    default=(' Real explanations can mix motives, so score this facet on its own terms '
             'regardless of whatever else the passage expresses.'))

# If notebook 05 shipped no single-facet definitions, this notebook does the
# preregistered joint pass and says so. It does not invent definitions: a split
# score taken against wording this notebook made up would not be comparable with
# the joint score, and the halo diagnostic compares exactly those two.
SPLIT_CALLS = SOLO_DEFN is not None
if not SPLIT_CALLS:
    print("  [skip] No single-facet definitions in rubric.json, so the separate-call "
          "desert/revenge passes are NOT run. The joint nine-facet pass (the "
          "preregistered one) runs as normal, and E4's halo diagnostic will report "
          "the split-call arm as not run.")

In [ ]:
# === ASSEMBLING ONE SINGLE-FACET PROMPT ======================================
# Deliberately parallel to build_user_prompt: same item-prompt context line, same
# context-only warning, same triple-quoted fence, same passage window -- so joint
# and split scores are read off the same passage under the same conditions and
# the halo delta isolates the one thing that changed.
def _solo_definition(facet):
    """Return (display name, definition) however notebook 05 wrote it down.

    Accepts either {'desert': 'the OFFENDER earned...'} or the two-part
    {'desert': ['desert', 'the OFFENDER earned...']} shape, so a reasonable
    change in notebook 05 does not silently drop a definition."""
    entry = SOLO_DEFN[facet]
    if isinstance(entry, (list, tuple)) and len(entry) >= 2:
        return entry[0], entry[1]
    if isinstance(entry, dict):
        return entry.get('name', facet), entry.get('definition', '')
    return facet, str(entry)

def build_solo_prompt(text, item_type, facet, mixed_anchor=True):
    name, defn = _solo_definition(facet)
    coocc = SOLO_COOCC if mixed_anchor else ''
    prompt_text = ITEM_PROMPTS.get(item_type, '')
    return (
        f'You are an expert annotator. Score ONLY the single facet "{name}" from 0.0 (absent) '
        f'to 1.0 (clearly, centrally present) for how strongly the passage expresses it.\n\n'
        f'Definition of {name}: {defn}\n{SOLO_TEACH}{coocc}\n\n'
        f'Prompt the writer answered (context only; do NOT let it bias the label): "{prompt_text}"\n\n'
        f'Passage:\n' + (chr(34) * 3) + f'\n{text[:TEXT_WINDOW]}\n' + (chr(34) * 3) + '\n\n'
        f'Return ONLY this JSON object: {{"{name}": 0.0}}'
    )

if SPLIT_CALLS:
    missing = [f for f in SPLIT_FACETS if f not in SOLO_DEFN]
    if missing:
        # Half a split pass is worse than none: the halo diagnostic needs both
        # sides of the pair, so if either is missing the arm does not run.
        print(f"  [skip] rubric.json has single-facet definitions for "
              f"{sorted(SOLO_DEFN)}, but not for {missing}. The separate-call "
              f"passes need both of {SPLIT_FACETS}, so they are not run.")
        SPLIT_CALLS = False
    else:
        print("Separate-call prompt ready. Example (revenge, no mixed anchor):\n")
        print(build_solo_prompt("An eye for an eye.", "forward", "revenge",
                                mixed_anchor=False)[:280] + " ...")

Separate-call prompt ready. Example (revenge, no mixed anchor):

You are an expert annotator. Score ONLY the single facet "revenge" from 0.0 (absent) to 1.0 (clearly, centrally present) for how strongly the passage expresses it.

Definition of revenge: retaliation, getting even, symmetric hitting-back: "payback", "eye for an eye", "serves them ...


In [ ]:
# === PARSING A REPLY THAT IS NOT TRUSTED TO BE CLEAN =========================
def parse_json_safe(text, prefill=''):
    """Extract a JSON object from a model reply, in escalating attempts.

    In order:
      1. put back a prefilled character if one was used. No current leg
         prefills, so every caller passes '' and this step is a no-op;
         it is kept so the parser can still cope if a leg ever does;
      2. strip markdown fences, which some models add despite the system prompt
         forbidding them;
      3. parse directly -- with the system prompt doing its job this is
         the normal path and the steps below are for the tail;
      4. fall back to the greedy first-'{'-to-last-'}' slice, which drops any
         prose that leaked in around the object;
      5. last resort, repair a truncation. max_tokens cutting a reply mid-key
         leaves a dangling '"censure": 0.' and NO closing brace anywhere, which
         is why step 4 cannot help: it needs a '}' to slice to. So close the
         object; if that still will not parse, drop the last comma-separated
         piece and try again, up to a dozen times.

    Returns the parsed dict, or None. None is a RESULT here, not an error: it
    becomes an explicit unscored row rather than a silent omission."""
    # An empty or None body (a refusal, a dropped connection, an empty content
    # block) is unparseable by definition -- return the None result rather than
    # letting json.loads raise inside a worker thread.
    if not text:
        return None
    # Step 1. Without this every Claude reply would fail to parse -- and it would
    # fail for EVERY response, not randomly, which is the signature to look for
    # in the failure log if one leg's parse rate comes back near zero.
    text = prefill + text
    # Step 2. A fenced object is a trivially recoverable failure and there is no
    # reason to throw the row away over it.
    text = re.sub(r'```(?:json)?\s*', '', text).replace('```', '')
    text = text.strip()

    def _obj(s):
        """Parse s, and accept it only if it is an OBJECT (a set of key: value
        pairs). A reply that parses to a list or a bare number is not a record,
        and returning it would raise later, inside a worker thread, where the
        row's identity is lost. It becomes the None result instead."""
        try:
            v = json.loads(s)
        except json.JSONDecodeError:
            return None
        return v if isinstance(v, dict) else None

    # Step 3. The clean path. Everything below exists for the tail, not the norm.
    got = _obj(text)
    if got is not None:
        return got
    first, last = text.find('{'), text.rfind('}')
    if first == -1:
        return None
    # Step 4. Greedy slice. first < last is checked so an unbalanced fragment
    # does not produce a backwards slice.
    if last > first:
        got = _obj(text[first:last + 1])
        if got is not None:
            return got
    # Step 5. HAZARD, and the reason max_tokens is generous: a repaired object is
    # missing whatever keys the truncation ate, and those stay NaN. NaN is
    # skipped by the ensemble mean, so a systematically truncated leg silently
    # contributes fewer raters to the tail facets rather than announcing itself.
    # Read the per-facet coverage printed at the end of this notebook, and
    # max_tokens, before reading anything into the agreement.
    # It is also what keeps a truncation costing ONE call and not five: at
    # temperature 0 every retry truncates in the same place, so a reply the
    # parser cannot repair burns the whole retry budget to buy the same failure.
    frag = text[first:]
    for _ in range(12):
        got = _obj(frag + '}')
        if got is not None:
            return got
        cut = frag.rfind(',')      # drop the last comma-separated piece
        if cut == -1:
            break
        frag = frag[:cut]
    return None

In [ ]:
# === MAPPING WHATEVER KEY CAME BACK ONTO A CANONICAL FACET NAME ==============
def normalize_label(s):
    """Map a returned key back to a canonical facet name, or None.

    Models paraphrase keys even under a literal schema: 'Moral Balance',
    'moral-balance', 'desert - the offender earned it'. Three attempts, in order
    of strictness: exact match after lowercasing and folding underscores and
    hyphens to spaces; then the prefix before ' - '; then a substring match.
    Returns None when nothing matches, and the caller DROPS the key rather than
    guessing -- a mis-mapped key would write one facet's score into another
    facet's column, which is worse than a missing value."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    s = str(s).strip().lower().replace('_', ' ').replace('-', ' ')
    if not s:
        return None
    s = ' '.join(s.split())
    # Built from FACETS, so the accepted vocabulary can never drift away from the
    # canonical list in rubric.json.
    canon = {f.replace('_', ' '): f for f in FACETS}
    if s in canon:                       # strictest: exact, space-folded
        return canon[s]
    if ' - ' in s:                       # 'desert - the offender earned it'
        prefix = s.split(' - ')[0].strip()
        if prefix in canon:
            return canon[prefix]
    # Loosest: the canonical name appears somewhere inside the returned key. Safe
    # only because the nine names are mutually non-containing, so no key can
    # match two facets.
    for k, f in canon.items():
        if k in s:
            return f
    return None

def _is_rate_limit(e):
    """True when an exception is the provider saying "too many requests".

    Matched on the class NAME and on the message rather than by catching a typed
    error, for two reasons. The anthropic package is imported only on the Claude
    leg, so naming its class inside an adapter would make that adapter unusable
    from any other leg the moment something dispatched it there. And the
    OpenAI-compatible shim in front of a third-party open-weight host does not
    always raise the SDK's own class. The stakes are low either way: a rate limit
    gets exponential backoff and every other error gets a one-second pause, so
    mistaking one for the other only changes how long a worker waits."""
    m = str(e).lower()
    return (type(e).__name__ == 'RateLimitError' or '429' in m
            or ('rate' in m and 'limit' in m))

# --- FAILURE ACCOUNTING ------------------------------------------------------
# FAILURE_LOG holds the VERBATIM text of the first few replies that could not be
# parsed, capped at 5. Small on purpose: it is a diagnostic sample, not an
# archive, and the cap keeps a systematic failure -- say, this leg refusing every
# graphic passage -- from filling the output with thousands of copies of the same
# message.
#
# It answers the question the counts cannot: WHY did parsing fail. Prose
# preamble, a refusal, a fence, a truncation and an authentication error all
# arrive as the same None, and they call for completely different fixes.
#
# Nothing is hidden by the cap. Every parse failure is an explicit null row in
# the .jsonl archive, and each of those rows now carries the reply text that
# failed, so a failure past the fifth is still readable in the file. This log is
# the one you can read without opening it.
FAILURE_LOG = []
_fail_lock = threading.Lock()

def _log_failure(model, prompt, last_raw):
    # last_raw is None when the failure was an exception rather than an
    # unparseable body, and those are skipped -- there is no text to inspect.
    with _fail_lock:
        if len(FAILURE_LOG) < 5 and last_raw is not None:
            FAILURE_LOG.append({'model': model, 'prompt_first120': prompt[:120],
                                'raw_response': last_raw[:500]})

# === PERMANENT VERSUS TRANSIENT, THE DISTINCTION THE RETRY LOOP WAS MISSING ===
# _is_rate_limit above answers "wait and try again?". This answers the opposite
# question: "will this EVER succeed?". A 4xx that is not a 429 is a statement
# about the request, not about the server's mood -- a model string that does not
# exist, a parameter the model no longer accepts, a key that is not valid. Every
# one of those was previously retried five times and then recorded as an unparsed
# row, which is why an unpinned roster surfaced as "0/20 parsed" instead of as
# "model not found". Matching on the status code rather than on the exception
# class keeps this working across SDK versions and across the two client
# libraries in this pipeline.
def _is_permanent(e):
    """True when retrying cannot possibly help.

    Two kinds qualify. A 4xx that is not a 429 is a statement about the
    REQUEST -- a model id that does not exist, a parameter this model no
    longer accepts, a key that is not valid. And a TypeError or
    AttributeError is a malformed CALL: no request ever left the machine,
    so there is nothing for the network to do differently next time.

    The second kind is not hypothetical. anthropic 1.0.0 removed the
    temperature parameter, and every Claude call died with
    TypeError: Messages.create() got an unexpected keyword argument
    'temperature' -- retried five times per row by the generic handler and
    then filed as an unparsed row, which is how it surfaced as "0/20
    parsed" rather than as the one sentence it actually was."""
    if isinstance(e, (TypeError, AttributeError)):
        return True
    code = getattr(e, 'status_code', None) or getattr(
        getattr(e, 'response', None), 'status_code', None)
    return isinstance(code, int) and 400 <= code < 500 and code != 429

print("_is_permanent defined (a 4xx that is not 429, or a malformed call, "
      "is never retried).")

_is_permanent defined (a 4xx that is not 429, or a malformed call, is never retried).


In [ ]:
# === LEG 1: the Anthropic adapter ============================================
# WHAT USED TO BE HERE, AND WHY IT IS NOT. Dark Side NB02 seeded the assistant
# turn with a single '{' so the model resumed already inside the JSON object and
# had no position from which to write "Here is the JSON:". That one character is
# what made near-100% parse rates achievable without a post-processor that
# guesses. Current Anthropic models reject a trailing assistant turn outright
# with a 400, so the technique is not available: it would fail every call on this
# leg rather than tightening it. The system prompt plus parse_json_safe carry the
# job now, and notebook 05's smoke test is what proves the parse rate holds
# BEFORE this notebook spends anything.
#
# THE RETRY SHAPE IS DELIBERATELY ASYMMETRIC, because the two failure kinds want
# opposite treatment:
#   parse failure   a short fixed pause (0.5s) and try again. At temperature 0
#                   the retry is near-identical, so this mostly buys a second
#                   chance at a transport-level truncation.
#   rate limit      exponential backoff, 1, 2, 4, 8, 16 seconds. Retrying a "429
#                   too many requests" immediately deepens the queue for all
#                   eight threads at once and turns a brief throttle into a
#                   cascade.
#   other exception one second, and give up on the final attempt.
# max_retries=4 means up to five attempts. Beyond that the failure is structural,
# not transient, and burning more calls on it only inflates the bill.
def claude_call(prompt, model_id, max_retries=4):
    """One Claude call. Returns the pair (reply text, parsed dict).

    The parsed half is None once the retry budget is spent; the text half is
    whatever last came back, or None if nothing ever did. BOTH halves are
    returned on every call because the archive keeps them side by side: the
    parsed record is what the analysis reads, and the reply text is the raw
    output prereg 6.7 asks be saved so the scoring can be repeated -- a parser
    fixed later can be re-run over the text, and cannot be re-run over a record
    the old parser already made.

    It NEVER raises: this runs inside a thread pool, and an exception escaping
    here would surface only when the future was read, taking the row's identity
    with it. None is the honest, recorded outcome."""
    last_raw = None
    for attempt in range(max_retries + 1):
        try:
            response = claude_client.messages.create(
                model=model_id,
                max_tokens=GEN_KW['max_tokens'],
                temperature=GEN_KW['temperature'],
                system=JSON_SYSTEM_PROMPT,
                # NO PREFILLED ASSISTANT TURN. Current Anthropic models reject a
                # trailing assistant turn with a 400, so the Dark Side prefill
                # discipline described above is no longer available: it would
                # fail every call on this leg rather than tightening it. The
                # system prompt plus parse_json_safe carry the job now, and
                # notebook 05's smoke test is what proves the parse rate holds
                # BEFORE this notebook spends anything.
                messages=[{'role': 'user', 'content': prompt}],
            )
            content = response.content[0].text
            last_raw = content
            # Token accounting for the running cost line: the provider's own
            # figures where it reports them, an estimate from length where it
            # does not, and never a silent zero.
            record_usage(getattr(response, 'usage', None), prompt, content,
                         'input_tokens', 'output_tokens')
            # prefill='' to match the call above: the reply now carries its own
            # opening brace. Passing '{' would prepend a second one and break
            # parsing for every response on this leg at once.
            parsed = parse_json_safe(content, prefill='')
            if parsed is not None:
                return last_raw, parsed
            time.sleep(0.5)
        except Exception as e:
            if _is_rate_limit(e):
                time.sleep(2 ** attempt)
            elif _is_permanent(e):
                # A PERMANENT ERROR IS NOT RETRIED. 404 model-not-found, 400
                # bad-request, 401 bad-key: none improve on the fifth attempt,
                # and at 2,800 rows retrying them is 14,000 pointless calls whose
                # only trace is a file full of unscored rows. Break on the first
                # one and let _log_failure carry the API's own sentence.
                _reason = f'<<permanent API error, not retried: {e!r}>>'[:400]
                _log_failure('claude', prompt, _reason)
                # The reason is returned AS the raw text so it lands in the
                # .jsonl archive beside the null record. FAILURE_LOG caps at
                # five and is a diagnostic sample; the archive is the record,
                # and a null row whose raw text is also null tells a later
                # reader nothing about why. The angle brackets mark the line as
                # ours rather than something a model said.
                return _reason, None
            elif attempt == max_retries:
                break
            else:
                time.sleep(1)
    _log_failure('claude', prompt, last_raw)
    return last_raw, None

In [ ]:
# === LEGS 2 AND 3: the two OpenAI-compatible adapters =========================
# WHY THE ADAPTERS ARE THIN. Three developers, three SDK shapes, one prompt. Each
# adapter does the same four things -- send the SAME prompt at temperature 0,
# read the text out of whatever envelope that SDK uses, parse, retry -- so the
# only thing that varies across legs is the model. Anything richer (a leg-specific
# reformat, a leg-specific repair) would put prompt variance into the agreement
# statistic.
#
# Two SDK affordances are used where they exist, and neither changes what is
# asked:
#   seed=DECODE_SEED            best-effort decoding determinism on top of
#                               temperature 0
#   response_format json_object server-side JSON enforcement, requested on the
#                               GPT leg only. It is the one hard guarantee of
#                               syntactic validity anywhere in the panel; the
#                               Anthropic and open-weight legs rely on the system
#                               prompt and on parse_json_safe
# prefill='' on both, as on every leg: they return a complete object.
def _openai_style_call(client, prompt, model_id, json_mode, tag, max_retries=4,
                       reasoning_model=False):
    """One call on an OpenAI-compatible endpoint. Returns (reply text, parsed
    dict) -- the same pair the Anthropic adapter returns, for the same reason:
    the archive keeps the model's own words beside the parser's reading of them,
    so the parse can be redone later and the record can be audited now."""
    last_raw = None
    for attempt in range(max_retries + 1):
        try:
            kw = dict(model=model_id, temperature=GEN_KW['temperature'],
                      seed=DECODE_SEED,
                      messages=[{'role': 'system', 'content': JSON_SYSTEM_PROMPT},
                                {'role': 'user', 'content': prompt}])
            # THE TOKEN-LIMIT PARAMETER HAS TWO NAMES, AND SENDING THE WRONG ONE
            # IS A 400 ON EVERY CALL. The GPT-5.x family rejects max_tokens
            # because it does not account for reasoning tokens; the open-weight
            # endpoint takes max_tokens and knows nothing about the other name.
            # Same value from GEN_KW either way -- one setting, two spellings.
            # reasoning_effort='none' rides along on the GPT leg because that
            # family accepts temperature ONLY at that effort, which is what keeps
            # this leg greedy alongside the other two.
            if reasoning_model:
                kw['reasoning_effort'] = GPT_REASONING_EFFORT
                kw['max_completion_tokens'] = GEN_KW['max_tokens']
            else:
                kw['max_tokens'] = GEN_KW['max_tokens']
            if json_mode:
                kw['response_format'] = {'type': 'json_object'}
            resp = client.chat.completions.create(**kw)
            content = resp.choices[0].message.content
            last_raw = content
            record_usage(getattr(resp, 'usage', None), prompt, content,
                         'prompt_tokens', 'completion_tokens')
            parsed = parse_json_safe(content, prefill='')
            if parsed is not None:
                return last_raw, parsed
            time.sleep(0.5)
        except Exception as e:
            if _is_rate_limit(e):
                time.sleep(2 ** attempt)
            elif _is_permanent(e):
                # Never retried -- see _is_permanent. At ~2,800 rows a retried
                # 400 is 14,000 pointless paid calls whose only trace is a file
                # of unscored rows.
                _reason = f'<<permanent API error, not retried: {e!r}>>'[:400]
                _log_failure(tag, prompt, _reason)
                return _reason, None
            elif attempt == max_retries:
                break
            else:
                time.sleep(1)
    _log_failure(tag, prompt, last_raw)
    return last_raw, None

def gpt_call(prompt, model_id):
    # Server-side JSON mode is available on this leg, so it is used.
    # reasoning_model=True: this leg is a GPT-5.x model, which needs
    # max_completion_tokens rather than max_tokens and needs reasoning_effort
    # 'none' before it will accept temperature at all.
    return _openai_style_call(gpt_client, prompt, model_id, True, 'gpt',
                              reasoning_model=True)

def open_call(prompt, model_id):
    # response_format is NOT requested here: hosted open-weight endpoints vary in
    # whether they support JSON mode, and asking for an unsupported option is an
    # error on some providers. parse_json_safe absorbs the difference, which is
    # exactly why the parser is written to be tolerant.
    return _openai_style_call(open_client, prompt, model_id, False, 'open')

# Dispatch table: model key -> its adapter. Keeps the scoring loop free of any
# per-leg branching, so adding a fourth rater is two lines (an adapter here, an
# entry in MODELS) and touches no analysis code.
CALL = {'claude': claude_call, 'gpt': gpt_call, 'open': open_call}
print(f"Adapters ready. This run uses: {CALL[MODEL_KEY].__name__}.")

Adapters ready. This run uses: claude_call.


In [ ]:
# === FROM A PARSED REPLY TO A CLEAN FACET RECORD ==============================
def coerce_facets(parsed):
    """Turn one parsed reply into a clean nine-facet + holistic record, or None.

    Starts from all-NaN over FACETS and fills only keys that normalize to a
    canonical facet, so an extra key the model invented is dropped and a facet
    the model omitted stays NaN (unmeasured) rather than becoming 0.0 (measured
    absent). The distinction matters: the ensemble mean skips NaN but averages in
    a 0.0. parsed is None -> return None, which becomes an explicit unscored row.
    Nothing here ever drops a response.

    This is a POST-PROCESSED record and the archive says so by keeping the reply
    text beside it: everything this function dropped, nulled or rounded is still
    recoverable from that text."""
    if parsed is None:
        return None
    # NaN, not 0.0. Collapsing the two would let a truncated or partial reply add
    # fake zeros to the ensemble mean and drag every facet toward the floor.
    out = {f: np.nan for f in FACETS}
    for k, v in parsed.items():
        canon = normalize_label(k)
        if canon is None:
            continue
        try:
            out[canon] = float(v)
        except (ValueError, TypeError):
            out[canon] = np.nan
    # The holistic call is validated against the allowed levels and set to None
    # otherwise, so the nominal agreement statistic downstream sees a clean
    # variable and an invented fifth category cannot become its own level. A leg
    # that answers outside the four levels every time therefore shows up as an
    # empty holistic column, which the coverage printout at the end reports on
    # its own line rather than folding into the facet counts.
    hol = str(parsed.get('holistic', '')).strip().lower()
    out['holistic'] = hol if hol in HOLISTIC_LEVELS else None
    # `evidence` is kept only when it is a dict. It is optional by design and is
    # carried for auditing; nothing downstream scores it, so a malformed evidence
    # field can never affect a facet value.
    out['evidence'] = parsed.get('evidence', {}) if isinstance(parsed.get('evidence'), dict) else {}
    return out

In [ ]:
# === THE TWO SCORERS THE RUN CALLS ===========================================
# Both return the PAIR (reply text, result). The text goes to the archive
# unaltered and the result goes to the table, so the file that reaches OSF holds
# what the model said as well as what this notebook made of it.
def score_one(text, item_type):
    """The joint nine-facet call for one passage on this run's model.

    Re-applies the word floor even though the corpus was already filtered, so a
    short text arriving by any route returns None instead of a fabricated
    score."""
    if not isinstance(text, str) or len(text.split()) < MIN_WORDS:
        return None, None
    raw, parsed = CALL[MODEL_KEY](build_user_prompt(text, item_type), MODEL_ID)
    return raw, coerce_facets(parsed)

def score_facet_solo(text, item_type, facet, mixed_anchor=True):
    """Score ONE facet in its own call. Returns (reply text, float or None).

    The reply is a one-key object, but the key still goes through normalize_label,
    because a model that answers {'Desert': 0.7} is right and would otherwise be
    discarded as a failure."""
    if not isinstance(text, str) or len(text.split()) < MIN_WORDS:
        return None, None
    prompt = build_solo_prompt(text, item_type, facet, mixed_anchor=mixed_anchor)
    raw, parsed = CALL[MODEL_KEY](prompt, MODEL_ID)
    if parsed is None:
        return raw, None
    # Pull the value out by canonical name rather than by position, and return
    # None if the key never matched or the value will not cast to a number. A
    # None here is an unscored cell the split-call mean skips -- it never becomes
    # a 0.0, which would read as "no revenge expressed" and bias the very
    # correlation this pass exists to estimate honestly.
    for k, v in parsed.items():
        if normalize_label(k) == facet:
            try:
                return raw, float(v)
            except (ValueError, TypeError):
                return raw, None
    return raw, None

print("Scorers defined. No API call has been made yet.")

Scorers defined. No API call has been made yet.


### The three passes, and the file they are written to

1. **joint** -- every response scored on all nine facets plus the holistic call.
   This is the preregistered pass (prereg 6.7).
2. **solo** -- desert and revenge again, one facet per call.
3. **solo_noanchor** -- the same two facets with the co-occurrence sentence
   removed, so the anchor's contribution can be measured rather than assumed.

Every call is written to `llm_scores_<MODEL_KEY>.jsonl` in batches, parse
failures included as explicit nulls: one line per response x item x pass, each
line carrying the model's own reply text and the record the parser made of it.
That file is the archive prereg 6.7 asks for, "all raw output saved so the
scoring can be repeated", and it is also what a resumed run reads to know what it
already owns.

**Parse failures are kept as null rows, never dropped.** The loss would not be
random: the long, graphic, vengeful replies are the ones most likely to fail, and
they carry the revenge and suffering signal. Dropping them would thin exactly the
tail the study measures while every printed count still looked healthy.

In [ ]:
# === THE WORK PLAN: every call this run has to make ==========================
# One flat list of tasks, each carrying everything the worker needs and the KEY
# that identifies it in the archive. Building the plan up front is what makes the
# resume possible: the key is what a finished call is recognised by.
JSONL_PATH = f"llm_scores_{MODEL_KEY}.jsonl"     # every call -> OSF
CSV_PATH   = f"llm_scores_{MODEL_KEY}.csv"       # the same thing as a table
# JSONL means "JSON lines": one self-contained JSON object per line. It is the
# right shape here, rather than a CSV, because each line is a NESTED object --
# nine facets, the holistic call, an optional evidence dictionary and the model's
# own reply text -- and because a file of lines can be appended to and read back
# a line at a time without rewriting what is already on disk. That is what makes
# checkpointing after every batch cheap enough to actually do.

def task_key(rid, item, kind, facet):
    """The identity of one call: which response, which question, which pass, and
    for the split passes which facet. Four parts, because the same response is
    called five times and the archive has to tell those five apart."""
    return (str(rid), str(item), str(kind), str(facet or ""))

TASKS = []
for _r in df_valid.itertuples(index=False):
    _txt = _r.text if isinstance(_r.text, str) else ""
    TASKS.append({'rid': _r.response_id, 'item': _r.item_type, 'text': _txt,
                  'kind': 'joint', 'facet': None})
    if SPLIT_CALLS:
        for _f in SPLIT_FACETS:
            # mixed anchor kept, then removed. Both, or the difference between
            # them -- the halo diagnostic -- cannot be computed.
            TASKS.append({'rid': _r.response_id, 'item': _r.item_type, 'text': _txt,
                          'kind': 'solo', 'facet': _f})
            TASKS.append({'rid': _r.response_id, 'item': _r.item_type, 'text': _txt,
                          'kind': 'solo_noanchor', 'facet': _f})
for _t in TASKS:
    _t['key'] = task_key(_t['rid'], _t['item'], _t['kind'], _t['facet'])

def prompt_for(t):
    """The exact text that will be sent for this task. Used by the workers and,
    below, to price the run before a penny is spent."""
    if t['kind'] == 'joint':
        return build_user_prompt(t['text'], t['item'])
    return build_solo_prompt(t['text'], t['item'], t['facet'],
                             mixed_anchor=(t['kind'] == 'solo'))

print(f"Work plan for the {MODEL_KEY} leg:")
print(f"  {len(df_valid):,} rows x "
      f"{1 + (4 if SPLIT_CALLS else 0)} calls each = {len(TASKS):,} calls.")
for _k in ('joint', 'solo', 'solo_noanchor'):
    _n = sum(t['kind'] == _k for t in TASKS)
    print(f"    {_k:14s} {_n:6,d}")

Work plan for the claude leg:
  59 rows x 5 calls each = 295 calls.
    joint              59
    solo              118
    solo_noanchor     118


In [ ]:
# === WHAT THIS RUN WILL COST, BEFORE IT SPENDS ANYTHING =======================
# A rough figure, from the length of the prompts actually about to be sent. Four
# characters per token is the usual English rule of thumb, and the reply is a
# small fixed object, so the estimate is close enough to catch the mistake that
# matters: a run that is about to cost ten times what was expected because the
# roster, the corpus or the number of passes is not what you thought.
#
# The two reply sizes are counted separately because they differ by a factor of
# twenty-five. A joint call returns nine facet numbers, a holistic word and
# optional evidence spans; a split call returns ONE number. Charging every call
# the joint rate would overstate a split-call run four times over, since four of
# every five calls in it are single numbers.
#
# The real figure, from the token counts the providers report, is printed as the
# run proceeds and again at the end.
_p = PRICES.get(MODEL_KEY, {'in': 0.0, 'out': 0.0})
_sample = TASKS[:min(50, len(TASKS))]
_avg_in_tok = (sum(len(prompt_for(t)) for t in _sample) / max(len(_sample), 1)) / 4
_JOINT_OUT_TOK, _SOLO_OUT_TOK = 260, 10     # nine facets and spans, or one number
_n_joint = sum(t['kind'] == 'joint' for t in TASKS)
_avg_out_tok = ((_n_joint * _JOINT_OUT_TOK
                 + (len(TASKS) - _n_joint) * _SOLO_OUT_TOK) / max(len(TASKS), 1))
_est = (len(TASKS) * _avg_in_tok / 1e6) * _p['in'] + \
       (len(TASKS) * _avg_out_tok / 1e6) * _p['out']
print(f"Rough cost of a complete {MODEL_KEY} run: about ${_est:,.2f} "
      f"({len(TASKS):,} calls, ~{_avg_in_tok:,.0f} prompt tokens each, "
      f"~{_avg_out_tok:,.0f} reply tokens each on average).")
print("That is an estimate from prompt length and the price table above, not a quote.")
if RUN_MODE == "smoke":
    print(f"Smoke mode is on, so this is the {df_valid['response_id'].nunique()}"
          f"-participant rehearsal: {len(df_valid):,} rows and {len(TASKS):,} "
          f"calls, not the corpus.")

Rough cost of a complete claude run: about $0.97 (295 calls, ~355 prompt tokens each, ~60 reply tokens each on average).
That is an estimate from prompt length and the price table above, not a quote.
Smoke mode is on, so this is the 20-participant rehearsal: 59 rows and 295 calls, not the corpus.


In [ ]:
# === MAKING THE CHECKPOINT SURVIVE A DISCONNECT (NOTEBOOK 06 ONLY) ===========
# WHY THIS EXISTS, AND WHY NOWHERE ELSE. Every other notebook in this chain is
# deliberately Drive-free: they hand CSVs to each other through WORKDIR, and a
# lost session costs a re-run of something free. This notebook is the exception.
# WORKDIR is a folder inside a Colab VM, the VM is discarded when the runtime is
# dropped, and the file it takes with it is llm_scores_<MODEL>.jsonl -- about an
# hour of paid calls per leg. The resume in the next cell reads exactly that
# file, so without a durable copy the resume protects against an interrupted
# cell and NOT against the disconnect it was written for.
#
# The download at the end of the notebook is not a substitute. It only fires if
# the run reaches the end; a runtime dropped at batch 180 of 232 leaves nothing
# to download.
#
# WHAT THIS DOES, IN BOTH DIRECTIONS
#   out  copy the archive to Drive every BACKUP_EVERY_BATCHES batches and once
#        when the run finishes
#   in   on startup, if Drive holds an archive for this leg and WORKDIR does
#        not, copy it back BEFORE the resume reads it. That half is what makes
#        the resume real.
#
# A FAILED MOUNT IS FATAL, ON PURPOSE. Falling back to no-backup would remove the
# protection silently, which is the exact failure this cell exists to prevent. If
# you are not on Colab, set BACKUP_TO_DRIVE = False and accept the original
# behaviour rather than getting a backup that is not happening.
import shutil as _shutil

DRIVE_OK = False
DRIVE_JSONL = DRIVE_CSV = None
_backup_fails = 0

if BACKUP_TO_DRIVE:
    try:
        from google.colab import drive as _gdrive
        _gdrive.mount('/content/drive')
        os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
        DRIVE_JSONL = os.path.join(DRIVE_BACKUP_DIR, JSONL_PATH)
        DRIVE_CSV   = os.path.join(DRIVE_BACKUP_DIR, CSV_PATH)
        DRIVE_OK = True
        print(f"Drive backup ON -> {DRIVE_BACKUP_DIR}")
    except Exception as _e:
        raise RuntimeError(
            f"BACKUP_TO_DRIVE is True but Drive could not be mounted "
            f"({type(_e).__name__}: {_e}).\n  This run would spend real money with "
            f"no durable copy of the archive, so it stops here.\n  Either fix the "
            f"mount, or set BACKUP_TO_DRIVE = False in the cell above and accept "
            f"that\n  an interrupted run loses everything since it started.")
else:
    print("Drive backup OFF. The archive lives only in this VM until the download "
          "at the end;\n  an interrupted run loses every call made since it "
          "started.")

def _lines_in(path):
    """Number of lines in a file, or 0 if it is not there."""
    try:
        with open(path, encoding='utf-8') as _fh:
            return sum(1 for _ in _fh)
    except Exception:
        return 0

# --- RESTORE, before the resume reads the archive ---------------------------
# Only when the local copy is ABSENT. A local archive is this session's own work
# and is never overwritten from Drive: silently replacing local calls with an
# older Drive copy would discard paid work and look like a successful resume.
if DRIVE_OK:
    _loc, _drv = _lines_in(JSONL_PATH), _lines_in(DRIVE_JSONL)
    if _drv and not os.path.exists(JSONL_PATH):
        _shutil.copy2(DRIVE_JSONL, JSONL_PATH)
        print(f"  restored {_drv:,} archived calls from the Drive copy. The resume "
              f"below will skip them.")
    elif _drv and _loc and _drv > _loc:
        print(f"  NOTE: Drive holds {_drv:,} lines and WORKDIR holds {_loc:,}. The "
              f"LOCAL file is used, because\n  this session may have appended to "
              f"it. If the local one is a stale partial upload, delete it and "
              f"re-run\n  this cell to restore the fuller Drive copy.")
    elif _loc:
        print(f"  {_loc:,} calls already in the local archive; Drive holds {_drv:,}.")

def backup_archive(also_csv=False):
    """Copy the archive to Drive. Never raises: a Drive hiccup must not kill a
    paid run mid-corpus, so a failure warns and the run carries on. Three in a
    row says the protection is gone and says it loudly."""
    global _backup_fails
    if not DRIVE_OK:
        return
    try:
        _shutil.copy2(JSONL_PATH, DRIVE_JSONL)
        if also_csv and os.path.exists(CSV_PATH):
            _shutil.copy2(CSV_PATH, DRIVE_CSV)
        _backup_fails = 0
    except Exception as _e:
        _backup_fails += 1
        print(f"  WARNING: Drive backup failed ({type(_e).__name__}: {_e}).")
        if _backup_fails >= 3:
            print("  *** THREE BACKUPS IN A ROW HAVE FAILED. The archive is no "
                  "longer protected. ***\n  Stop the run, fix Drive, and re-run: "
                  "the resume will skip everything already on disk.")

print("backup_archive defined." if DRIVE_OK else "backup_archive defined (no-op: backup is off).")


Mounted at /content/drive
Drive backup ON -> /content/drive/MyDrive/retribution_nlp_archive
backup_archive defined.


In [ ]:
# === RESUME: what is already paid for ========================================
# On startup the archive is read and every call already in it is struck off the
# plan. This is what makes a dropped Colab runtime an inconvenience rather than a
# second bill. Nothing is re-sent, and the notebook says exactly how much it
# skipped.
def read_archive(path=JSONL_PATH, model_id=None):
    """Every record on disk written by THIS pinned model version.

    Records from a different version string are returned separately, not thrown
    away: they are real scores from a real run, they just are not this rater."""
    model_id = MODEL_ID if model_id is None else model_id
    mine, others, broken = [], 0, 0
    if not os.path.exists(path):
        return mine, others, broken
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                # A VM killed mid-write leaves one truncated last line. Ignore it
                # and carry on -- the call it belonged to simply gets made again.
                broken += 1
                continue
            if rec.get("model_id") != model_id:
                others += 1
                continue
            mine.append(rec)
    return mine, others, broken

_prior, _other_ver, _broken = read_archive()
already, n_null_prior = set(), 0
for _rec in _prior:
    _is_null = _rec.get("raw") is None
    n_null_prior += _is_null
    # A previously failed call is treated as finished unless RETRY_FAILED is on.
    if _is_null and RETRY_FAILED:
        continue
    already.add(task_key(_rec.get("rid"), _rec.get("item"),
                         _rec.get("pass_kind"), _rec.get("facet")))

todo = [t for t in TASKS if t["key"] not in already]
print(f"Archive: {len(_prior):,} calls already on disk for {MODEL_ID}.")
print(f"  skipping {len(TASKS) - len(todo):,} calls that are already paid for")
print(f"  still to run: {len(todo):,} calls")
if n_null_prior:
    print(f"  {n_null_prior:,} of the finished calls came back unparseable and are "
          f"kept as null rows (RETRY_FAILED={RETRY_FAILED}).")
if _other_ver:
    print(f"  {_other_ver:,} lines in the file came from a DIFFERENT version string "
          f"and are ignored here. Scores from two checkpoints are two raters, not one.")
_diff_rubric = sum(1 for r in _prior
                   if r.get("rubric_fingerprint") not in (None, RUBRIC_FINGERPRINT))
if _diff_rubric:
    print(f"  {_diff_rubric:,} of them were scored under a DIFFERENT codebook "
          f"fingerprint. They are kept and counted as finished. If the change to "
          f"rubric.json was a real one, move {JSONL_PATH} aside and run again, so "
          f"the whole corpus is scored under one instruction.")
if _broken:
    print(f"  {_broken} line(s) were unreadable, most likely a run cut off mid-write. "
          f"Those calls will simply be made again.")

Archive: 0 calls already on disk for claude-opus-4-6.
  skipping 0 calls that are already paid for
  still to run: 295 calls


In [ ]:
# === ONE CALL, ONE ARCHIVE RECORD ============================================
def _jsonable(v):
    """NaN is not valid JSON. Every unmeasured facet is written as null instead,
    so the archive can be read by any tool in any language and an unmeasured
    facet still says "unmeasured" rather than becoming a number."""
    if isinstance(v, float) and (np.isnan(v) or np.isinf(v)):
        return None
    if isinstance(v, dict):
        return {k: _jsonable(x) for k, x in v.items()}
    return v

def run_task(t):
    """Make one call and return the record that goes in the archive.

    It never raises. A bug or an unexpected reply shape becomes a recorded null
    with its error text attached, because a single bad row must not throw away a
    batch of calls that have already been paid for."""
    err, reply = None, None
    try:
        if t['kind'] == 'joint':
            reply, raw = score_one(t['text'], t['item'])
        else:
            reply, raw = score_facet_solo(t['text'], t['item'], t['facet'],
                                          mixed_anchor=(t['kind'] == 'solo'))
    except Exception as e:
        raw, err = None, f"{type(e).__name__}: {e}"
    # One record per call, keyed by response, item and pass, carrying BOTH:
    #   raw_reply  the model's own text, exactly as it arrived. This is the raw
    #              output prereg 6.7 asks be saved "so the scoring can be
    #              repeated": a parser corrected later can be re-run over it,
    #              and a key this run's normalize_label rejected is still there
    #              to be recovered.
    #   raw        the coerced record: nine canonical facets, a validated
    #              holistic level, evidence if it was a dictionary. Null where
    #              the reply could not be parsed at all.
    return {'rid': t['rid'], 'item': t['item'],
            'model': MODEL_KEY, 'model_id': MODEL_ID,
            'rubric_fingerprint': RUBRIC_FINGERPRINT,
            'pass_kind': t['kind'], 'facet': t['facet'],
            'raw': _jsonable(raw), 'raw_reply': reply, 'error': err,
            'run_mode': RUN_MODE,
            'at': datetime.datetime.now().isoformat(timespec='seconds')}

In [ ]:
# === WRITING A BATCH DOWN, SAFELY ============================================
def _close_off_last_line(path):
    """Make sure the archive ends with a line break before anything is appended.

    A run cut off mid-write can leave its last line without one. If the next
    batch appended straight onto it, two records would be welded into a single
    unreadable line and the good one would be lost along with the broken one --
    a paid call thrown away by a stray missing character."""
    if os.path.exists(path) and os.path.getsize(path):
        with open(path, "rb+") as fh:
            fh.seek(-1, os.SEEK_END)
            if fh.read(1) != b"\n":
                fh.write(b"\n")

def append_records(recs):
    """Append a batch to the archive and force it to disk.

    flush() moves the bytes out of Python's buffer; fsync() makes the operating
    system write them down. Without both, a batch can sit in a buffer inside a VM
    that is about to be discarded, which is precisely the case this whole design
    exists to survive. (fsync guarantees the VM's own disk and nothing more,
    which is why the archive is also downloaded off the machine at the end.)

    default=str keeps a stray value that JSON has no representation for -- a
    numpy float, a timestamp -- from raising mid-write and taking a batch of
    paid calls with it. It is written as text rather than lost."""
    _close_off_last_line(JSONL_PATH)
    with open(JSONL_PATH, "a", encoding="utf-8") as fh:
        for r in recs:
            fh.write(json.dumps(r, default=str) + "\n")
        fh.flush()
        os.fsync(fh.fileno())

In [ ]:
# === THE RUN: batches of calls, written down after every batch ===============
# CONCURRENCY. Eight calls in flight inside each batch, using the future -> index
# pattern: as_completed hands futures back in COMPLETION order, which is not
# input order, so the dictionary maps each future to the position it came from
# and out[i] holds the record for batch item i. Every record also carries its own
# response id, item and pass, so nothing downstream depends on row order -- that
# belt-and-braces is deliberate, because in the frame-based version of this code
# appending in completion order would have silently shuffled every score against
# its response and produced a fully populated, entirely meaningless table.
#
# CHECKPOINT AFTER EVERY BATCH. The archive is appended and forced to disk before
# the next batch starts. A run that dies at any point keeps everything up to the
# last completed batch, and the next run skips it.
_t0 = time.time()
_n_batches = (len(todo) + BATCH_SIZE - 1) // BATCH_SIZE
_n_done = _n_err = 0
if not todo:
    print("Nothing to do: every call for this model is already in the archive.")
_bar = tqdm(total=len(todo), desc=f"{MODEL_KEY} calls", disable=not todo)
for _b, _start in enumerate(range(0, len(todo), BATCH_SIZE), start=1):
    _batch = todo[_start:_start + BATCH_SIZE]
    _out = [None] * len(_batch)
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as _ex:
        _futs = {_ex.submit(run_task, t): i for i, t in enumerate(_batch)}
        for _fut in as_completed(_futs):
            _out[_futs[_fut]] = _fut.result()
    append_records(_out)                       # <- the checkpoint
    # ...and off the machine, every BACKUP_EVERY_BATCHES batches. The local
    # append above is what the resume reads; this is what survives the VM.
    if _b % BACKUP_EVERY_BATCHES == 0 or _b == _n_batches:
        backup_archive()
    _n_done += len(_out)
    _n_err += sum(r['error'] is not None for r in _out)
    _bar.update(len(_batch))
    _bar.set_postfix_str(f"${cost_so_far():,.2f}")
    if _b % PRINT_EVERY == 0 or _b == _n_batches:
        _el = time.time() - _t0
        _rate = _n_done / max(_el, 1e-9)
        _left = (len(todo) - _n_done) / max(_rate, 1e-9)
        print(f"  [checkpoint {_b}/{_n_batches}] {_n_done:,}/{len(todo):,} calls on disk "
              f"| cost so far {usage_line()} "
              f"| projected total ${cost_so_far() / max(_n_done, 1) * len(todo):,.2f} "
              f"| about {_left / 60:,.0f} min left")
_bar.close()
print(f"\nRun finished: {_n_done:,} new calls, {usage_line()} spent this session "
      f"({USAGE['in']:,} prompt tokens, {USAGE['out']:,} reply tokens).")
if USAGE['estimated']:
    print(f"  {USAGE['estimated']:,} of those calls came back with no token count, so "
          f"their share of the cost is estimated from prompt and reply length rather "
          f"than billed figures. Treat the total as approximate, not as an invoice.")
if _n_err:
    print(f"  {_n_err:,} call(s) raised an error and were recorded as nulls with the "
          f"message attached. If that is a large share of the run, stop and read one: "
          f"a systematic error is a wrong version string or an unsupported option, "
          f"not bad luck.")

claude calls:   0%|          | 0/295 [00:00<?, ?it/s]

  [checkpoint 5/5] 295/295 calls on disk | cost so far $0.88 | projected total $0.88 | about 0 min left

Run finished: 295 new calls, $0.88 spent this session (132,797 prompt tokens, 8,558 reply tokens).


In [ ]:
# === WHY ANY CALL FAILED =====================================================
# Nothing prints here when every reply parsed. When something did fail, this is
# the first thing to read. What to look for:
#   prose before the object   the system prompt is not landing
#   a refusal                 the research-data framing is not landing, and the
#                             loss will be correlated with violent content, which
#                             is the one kind of missingness this study cannot
#                             afford
#   a truncated object        the parser repairs what it can and the facets the
#                             truncation ate stay unmeasured -- raise max_tokens
#   an authentication or quota message wearing a parse failure's clothes
# This is a sample of five. Every failed reply is in llm_scores_<MODEL_KEY>.jsonl
# under raw_reply, so nothing here is the only copy of anything.
if FAILURE_LOG:
    print(f"First {len(FAILURE_LOG)} unparseable replies from this session "
          f"(a diagnostic sample, capped at 5):")
    for _j, _fl in enumerate(FAILURE_LOG):
        print(f'  [{_j + 1}] model={_fl["model"]}')
        print(f'      prompt: "{_fl["prompt_first120"]}"')
        print(f'      reply : "{_fl["raw_response"][:200]}"')
else:
    print("No unparseable replies in this session.")

No unparseable replies in this session.


## Save and check

In [ ]:
# === READ THE ARCHIVE BACK, AND SCREEN IT ====================================
# The .jsonl is the record of every CALL; the .csv is the record of every
# RESPONSE. Both are written, because they answer different questions: the
# archive answers "what exactly did this model say", the table answers "what is
# this model's score for this response".
#
# TWO THINGS THIS SCREEN DECIDES, both stated rather than left implicit:
#
#   a line that cannot be placed   a record missing its response id, its item or
#       its pass cannot be attached to any row. The resume above already
#       tolerates such a line (it reads the missing field as None and simply
#       re-runs that call), so this must not die on one AFTER the run has been
#       paid for. Counted, skipped, reported.
#
#   the same call recorded twice   the archive can legitimately hold two records
#       for one call: with RETRY_FAILED=True a resumed run re-runs a call that
#       failed before and appends a second line. THE LATER LINE WINS, which is
#       exactly what a retry is for, and that is the policy rather than an
#       accident of dictionary order. It is counted and printed, because a large
#       count means something else happened -- a hand-edited file, or two runs
#       whose records were both kept.
_recs, _, _ = read_archive()
_KINDS = ('joint', 'solo', 'solo_noanchor')
_unplaceable, _repeat_calls, _seen, _usable = 0, 0, set(), []
for _rec in _recs:
    _rid, _item, _kind = _rec.get('rid'), _rec.get('item'), _rec.get('pass_kind')
    # A split record with no facet has no column to go in, so it is unplaceable
    # for the same reason a record with no response id is.
    if _rid is None or _item is None or _kind not in _KINDS or (
            _kind != 'joint' and not _rec.get('facet')):
        _unplaceable += 1
        continue
    _tk = task_key(_rid, _item, _kind, _rec.get('facet'))
    if _tk in _seen:
        _repeat_calls += 1
    _seen.add(_tk)
    _usable.append(_rec)

print(f"Archive: {len(_recs):,} records for {MODEL_ID}, {len(_usable):,} usable.")
if _unplaceable:
    print(f"  {_unplaceable:,} record(s) carry no response id, item or pass and "
          f"cannot be placed in the table. They are left in the .jsonl and "
          f"ignored here; the calls they stood for were re-made above.")
if _repeat_calls:
    print(f"  {_repeat_calls:,} call(s) appear more than once in the archive. The "
          f"LAST record of each wins, which is what RETRY_FAILED=True is for. If "
          f"you did not re-run failures, read the file before trusting the table.")
_fp_other = sum(1 for r in _usable
                if r.get('rubric_fingerprint') not in (None, RUBRIC_FINGERPRINT))
if _fp_other:
    print(f"  {_fp_other:,} of them were scored under a DIFFERENT codebook "
          f"fingerprint and are folded in regardless. Two codebooks are two "
          f"instructions, not one rater: move {JSONL_PATH} aside and re-run if the "
          f"change to rubric.json was a real one.")

Archive: 295 records for claude-opus-4-6, 295 usable.


In [ ]:
# === FOLD THE ARCHIVE INTO ONE ROW PER RESPONSE x ITEM =======================
# Column names carry the model key -- desert__claude, holistic__claude -- so that
# when the three files are pooled downstream no column can be mistaken for
# another rater's. Keeping every rater individually addressable all the way to
# the export is what makes the shared-error diagnostic possible at all;
# collapsing to a mean earlier would throw away the only evidence that could
# reveal a blind spot the models share.
#
# Every field is read with .get(), never with [], for the same reason the screen
# above exists: this cell runs after the money is spent, and a KeyError here
# would cost the table, not the calls.
rows = {}
for _rec in _usable:
    _k = (str(_rec.get('rid')), str(_rec.get('item')))
    _row = rows.setdefault(_k, {'response_id': _rec.get('rid'),
                                'item_type': _rec.get('item')})
    _raw = _rec.get('raw')
    _kind = _rec.get('pass_kind')
    if _kind == 'joint':
        # A null result becomes a full row of NaN plus an empty holistic -- the
        # explicit unscored row. It holds its place, and it contributes nothing
        # to any mean or agreement statistic downstream.
        for _f in FACETS:
            _v = None if not isinstance(_raw, dict) else _raw.get(_f)
            try:
                _row[f'{_f}__{MODEL_KEY}'] = np.nan if _v is None else float(_v)
            except (TypeError, ValueError):
                _row[f'{_f}__{MODEL_KEY}'] = np.nan
        _row[f'holistic__{MODEL_KEY}'] = (_raw.get('holistic')
                                          if isinstance(_raw, dict) else None)
    else:
        # The split columns are named distinctly and never merged into the joint
        # ones. The joint estimate is the preregistered one and must stay
        # reportable in its own right; the split estimate carries the primary
        # weight for E4; and the DIFFERENCE between them is the halo diagnostic.
        # Averaging them together would destroy the only quantity that can tell a
        # real co-occurrence from a rating artifact.
        _suffix = 'split' if _kind == 'solo' else 'split_noanchor'
        _col = f"{_rec.get('facet')}_{_suffix}__{MODEL_KEY}"
        try:
            _row[_col] = np.nan if _raw is None else float(_raw)
        except (TypeError, ValueError):
            _row[_col] = np.nan
print(f"Folded {len(_usable):,} archived calls into {len(rows):,} response rows.")

Folded 295 archived calls into 59 response rows.


In [ ]:
# === ATTACH THE RESPONSE METADATA AND WRITE THE TABLE ========================
scores = pd.DataFrame(list(rows.values()))

# The join is on the composite key response_id x item_type, never on response_id
# alone: one participant has up to three texts and up to three different desert
# scores. It is a join and not a side-by-side concatenation, because the archive
# is keyed and can be read back in any order -- position means nothing in a file
# that a resumed run appended to out of sequence.
_meta_cols = [c for c in ['response_id', 'item_type', 'orientation', 'vignette', 'n_words']
              if c in df_valid.columns]
meta = df_valid[_meta_cols].copy()
meta['response_id'] = meta['response_id'].astype(str)
scores['response_id'] = scores['response_id'].astype(str)

# A merge can only multiply rows if the frame being merged IN repeats a key. That
# is the frame checked here. In this notebook it holds because the fold above
# built it from a dictionary keyed on exactly this pair; it is asserted anyway,
# because that construction is one edit away from changing and because every
# notebook in the chain checks the same contract at the same point.
assert not scores.duplicated(['response_id', 'item_type']).any(), (
    f"The folded score frame repeats "
    f"{int(scores.duplicated(['response_id', 'item_type']).sum())} "
    f"(response_id, item_type) pair(s). Merging it would count those responses "
    f"twice in every downstream analysis.")
# The LEFT frame is where a duplicate could really come from, and it is checked
# where it is loaded, against openends_long.csv itself, in the cell headed LOAD
# THE CORPUS above.
out = meta.merge(scores, on=['response_id', 'item_type'], how='left')
# A backstop and not a detector, said plainly: a left join cannot lose rows, and
# it can only add them when the frame merged in repeats a key -- which the line
# above has just ruled out. This fires only if that assertion is removed or the
# join is ever changed to an outer one, and it is kept for exactly that case,
# because the row count is what every notebook in this chain protects here and a
# future edit should be caught by a stopped notebook rather than by a reader.
assert len(out) == len(df_valid), (
    f"Row count changed in the merge ({len(df_valid)} -> {len(out)}): the frame "
    f"of scores repeats a (response_id, item_type) pair.")

# A stable column order: metadata, the nine joint facets, the holistic call, then
# the split columns. Downstream reads by name, but a predictable file is easier
# for a human to check.
_facet_cols = [f'{f}__{MODEL_KEY}' for f in FACETS] + [f'holistic__{MODEL_KEY}']
# When the separate-call pass did not run, its columns are left OUT of the file
# rather than written empty. An empty column reads as "the model failed on every
# row", which is a different and much worse thing than "this pass was not run".
# The printout below says which of the two happened.
_split_cols = ([f'{f}_{s}__{MODEL_KEY}' for f in SPLIT_FACETS
                for s in ('split', 'split_noanchor')] if SPLIT_CALLS else [])
for _c in _facet_cols + _split_cols:
    if _c not in out.columns:
        # A column no record filled is still written, so the shape of the file
        # does not depend on how many calls came back.
        out[_c] = np.nan
out = out[_meta_cols + _facet_cols + _split_cols]
print(f"Table built: {out.shape[0]:,} rows x {out.shape[1]} columns.")

Table built: 59 rows x 19 columns.


In [ ]:
# === WRITE IT, ONCE THE CONTRACT IS PROVEN ===================================
# The last check before the file leaves this notebook: one row per response x
# item. Notebook 07 merges three of these files on that pair, notebook 10 and
# notebook 11 join survey scales onto the result, and a repeated pair there would
# multiply rows in every one of them -- inflating the sample size of every
# exploratory test while the printed participant count stayed at 948.
# WHAT MAKES IT FIRE: a duplicate in openends_long.csv (caught earlier), or a
# fold that stopped being one-row-per-key.
_dups_out = out.duplicated(['response_id', 'item_type'], keep=False)
assert not _dups_out.any(), (
    f"{int(_dups_out.sum())} rows of the finished table share a (response_id, "
    f"item_type) pair, for example "
    f"{out.loc[_dups_out, ['response_id', 'item_type']].head(3).to_dict('records')}. "
    f"This file is merged on that pair by notebooks 07, 10 and 11 and must not "
    f"fan out. Nothing is written.")

# Replacing a full table with a rehearsal is recoverable but worth saying out
# loud: the archive keeps every call, so a later full run resumes over it and
# pays only for the remainder, but until then the file notebook 07 reads is a
# 20-participant rehearsal.
_prev = _load_manifest()["files"].get(CSV_PATH, {})
if RUN_MODE == "smoke" and _prev.get("run_mode") == "full":
    print(f"  WARNING: {CSV_PATH} on disk was written by a FULL run "
          f"({_prev.get('rows')} rows, {_prev.get('written_at')}), and this smoke "
          f"run is about to replace it with {len(out):,} rows. No paid call is "
          f"lost -- set RUN_MODE = \"full\" and run again to rebuild it.")

out.to_csv(CSV_PATH, index=False)
print(f"Written: {CSV_PATH}")

Written: llm_scores_claude.csv


In [ ]:
# === COVERAGE: HOW MUCH OF THE CORPUS THIS RATER ACTUALLY SCORED =============
# WHAT THIS PRINTOUT MEANS. Per facet, how many responses came back carrying a
# number. Expect every line within a percent or two of the row count. A leg
# sitting far below the others is not a noisier rater -- it is a broken one
# (wrong prefill, wrong version string, JSON mode unsupported, a refusal
# pattern), and the failure log above says which. Reading a low-coverage leg's
# disagreement as disagreement would be wrong.
#
# WHY PER FACET AND NOT ONE PROBE FACET. A truncated reply still parses after
# repair, and what it loses is always the LAST keys of the object. Checking one
# facet and calling the answer "responses scored" would report full coverage for
# a leg that systematically lost the tail of every reply. If these counts slope
# downwards, the facets at the end of the schema are the ones being eaten: raise
# max_tokens and re-run those calls.
#
# THE HOLISTIC LINE IS SEPARATE because it can fail on its own. It is filled only
# when the model's word is one of the four levels in rubric.json, so a leg that
# answers "unclear" every time scores every facet and no holistic at all. A
# holistic count below the facet counts means one of two things, and they are
# told apart by whether the last facets dropped with it: a vocabulary mismatch
# if the facets are all full, a truncation if they are not.
_n_rows = len(out)
_facet_n = {_f: int(out[f'{_f}__{MODEL_KEY}'].notna().sum()) for _f in FACETS}
_any_n = (int(out[[f'{_f}__{MODEL_KEY}' for _f in FACETS]].notna().any(axis=1).sum())
          if _n_rows else 0)
_hol_n = int(out[f'holistic__{MODEL_KEY}'].notna().sum()) if _n_rows else 0
_best = max(_facet_n.values()) if _facet_n else 0
print(f"Joint pass: {_any_n:,}/{_n_rows:,} responses returned a usable object "
      f"({_n_rows - _any_n:,} unscored rows kept as blanks).")
print("Responses carrying a number, per facet:")
for _f in FACETS:
    _flag = "   <- well below the others" if _facet_n[_f] < 0.9 * _best else ""
    print(f"  {_f:18s} {_facet_n[_f]:6,d}/{_n_rows:,}{_flag}")
print(f"  {'holistic':18s} {_hol_n:6,d}/{_n_rows:,}"
      + ("   <- either this leg's words are not the four levels in rubric.json, "
         "or its replies are being cut off before the holistic key"
         if _hol_n < 0.9 * _any_n else ""))
if SPLIT_CALLS:
    for _f in SPLIT_FACETS:
        _a = int(out[f'{_f}_split__{MODEL_KEY}'].notna().sum())
        _b = int(out[f'{_f}_split_noanchor__{MODEL_KEY}'].notna().sum())
        print(f"Separate-call pass, {_f}: {_a:,} with the mixed anchor, "
              f"{_b:,} without it.")
else:
    print("Separate-call passes: not run (no single-facet definitions in rubric.json).")

Joint pass: 59/59 responses returned a usable object (0 unscored rows kept as blanks).
Responses carrying a number, per facet:
  desert                 59/59
  proportionality        59/59
  censure                59/59
  moral_balance          59/59
  revenge                59/59
  suffering              59/59
  deterrence             59/59
  incapacitation         59/59
  rehabilitation         59/59
  holistic               59/59
Separate-call pass, desert: 59 with the mixed anchor, 59 without it.
Separate-call pass, revenge: 59 with the mixed anchor, 59 without it.


In [ ]:
# === RECORD BOTH FILES IN THE MANIFEST =======================================
# done() is called only after the table is on disk and its one-row-per-response
# contract has been asserted, so the manifest can never advertise a file that is
# not there or a shape that was never checked.
_note = (f"model_key={MODEL_KEY}; model_id={MODEL_ID}; "
         f"rubric={RUBRIC_FINGERPRINT}; temperature={GEN_KW['temperature']}; "
         f"passes=joint" + ("+solo+solo_noanchor" if SPLIT_CALLS else "") +
         f"; calls_in_archive={len(_recs)}; session_cost_usd={cost_so_far():.2f}"
         + (f"; {USAGE['estimated']} call(s) had no token count and were estimated"
            if USAGE['estimated'] else ""))
done(JSONL_PATH, rows=len(_recs),
     note="one line per call: the model's reply text (raw_reply) beside the "
          "parsed record (raw), nulls kept. " + _note)
done(CSV_PATH, rows=_n_rows, note="one row per response_id x item_type. " + _note)

  [wrote] llm_scores_claude.jsonl  (295 rows, 126,056 bytes)
  [wrote] llm_scores_claude.csv  (59 rows, 6,440 bytes)


In [ ]:
# === TAKE THE ARCHIVE OFF THE MACHINE ========================================
# WHY THIS STILL MATTERS WITH THE DRIVE BACKUP ON. WORKDIR is a folder inside a
# Colab VM. The flush and fsync in
# append_records guarantee the bytes reached THAT machine's disk -- and that
# machine is discarded when the runtime ends. Until notebook 12 gathers the
# bundle, this .jsonl and its several thousand paid calls exist in exactly one
# place. Downloading it after every leg is what makes "a re-run picks up where
# the last one stopped" true across a disconnect and not only within a session.
# Keep the file: dropping it back into WORKDIR before a re-run is what saves
# buying the same scores twice.
# One last copy to Drive first, including the CSV this time, so the durable copy
# matches the finished run rather than the last checkpoint before it.
backup_archive(also_csv=True)
if DRIVE_OK:
    print(f"Final copy on Drive: {DRIVE_BACKUP_DIR}")

try:
    from google.colab import files
    files.download(JSONL_PATH)
    print(f"{JSONL_PATH} sent to your browser's downloads. Keep it: it is the "
          f"only copy outside this virtual machine, and re-buying it costs the "
          f"full price of this leg again.")
except Exception as _e:
    # Not running in Colab, or the browser refused the download. Say so and carry
    # on -- the file is on disk either way, and stopping here would help nobody.
    print(f"  [note] could not hand {JSONL_PATH} to the browser ({type(_e).__name__}). "
          f"Download it by hand from the file browser in the left sidebar BEFORE "
          f"this runtime disconnects. It is the archive of every paid call.")

Final copy on Drive: /content/drive/MyDrive/retribution_nlp_archive


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

llm_scores_claude.jsonl sent to your browser's downloads. Keep it: it is the only copy outside this virtual machine, and re-buying it costs the full price of this leg again.


In [ ]:
# === WHICH RATERS ARE SCORED, AND WHICH ARE STILL OUTSTANDING ================
# Read from the manifest rather than from the folder, so a stray file someone
# copied in cannot be mistaken for a finished leg.
_m = _load_manifest()
scored_models, outstanding = [], []
for _k in MODELS:
    _rec = _m["files"].get(f"llm_scores_{_k}.csv")
    (scored_models if _rec else outstanding).append((_k, _rec))

print("\n" + "=" * 72)
print(f"NOTEBOOK {NB_ID} ({NB_NAME}) finished for the {MODEL_KEY} leg.")
print("=" * 72)
print("\nScored so far:")
for _k, _rec in scored_models:
    print(f"  {_k:8s} {_rec.get('rows')} rows, written {_rec.get('written_at')}, "
          f"mode {_rec.get('run_mode')}")
    print(f"           {_rec.get('note', '')}")
if outstanding:
    print("\nStill outstanding:")
    for _k, _ in outstanding:
        print(f"  {_k:8s} -> {MODELS[_k]}")
    _next = outstanding[0][0]
    print(f"\nWhat to do next: go back to the cell that sets MODEL_KEY, change it "
          f"to \"{_next}\", and run this notebook again. Nothing else changes, and "
          f"the {MODEL_KEY} files you just wrote are left alone.")
else:
    print("\nAll of the raters on the roster have been scored.")
    print("The next notebook pools llm_scores_*.csv into the ensemble mean and the "
          "holistic majority, and computes the agreement between the raters.")
if any(_rec.get("run_mode") == "smoke" for _, _rec in scored_models if _rec):
    print("\nNOTE: at least one leg above was written in smoke mode, which scores "
          "only the first 20 participants (up to 60 rows, five calls each). Set "
          "RUN_MODE = \"full\" and run it again; the calls already paid for will "
          "be skipped.")


NOTEBOOK 06 (score_one_model) finished for the claude leg.

Scored so far:
  claude   59 rows, written 2026-08-26T21:19:00, mode smoke
           one row per response_id x item_type. model_key=claude; model_id=claude-opus-4-6; rubric=fef0381e46892391; temperature=0.0; passes=joint+solo+solo_noanchor; calls_in_archive=295; session_cost_usd=0.88

Still outstanding:
  gpt      -> gpt-5.6-terra
  open     -> meta-llama/Llama-3.3-70B-Instruct-Turbo

What to do next: go back to the cell that sets MODEL_KEY, change it to "gpt", and run this notebook again. Nothing else changes, and the claude files you just wrote are left alone.

NOTE: at least one leg above was written in smoke mode, which scores only the first 20 participants (up to 60 rows, five calls each). Set RUN_MODE = "full" and run it again; the calls already paid for will be skipped.
